# AKANE E16-0：Gemma は「逆向きの問い」に弱いか

同じ「AはBの一種」を、2つの向きの4択で聞きます。
- 前向き：「Abolitionist is a type of which? A. person B. animal …」（物 → 分類）
- 逆向き：「Which is a type of person? A. abolitionist B. llama …」（分類 → 物）
- 組は 514（物 501、分類 46）。外れの選択肢はやさしい（でたらめ）と難しい（兄弟の分類）の2通り。正解の位置は A〜D に1回ずつ。
- おまけ：答えを自由に書かせる版（前向き「〜は何の一種？」、逆向き「〜の種類を20挙げて」）。
- 手で書いた語の一覧は使っていません。

## 手順
1. 「ランタイム」→「ランタイムのタイプを変更」→ **L4 GPU**（T4でも動きます）。
2. 鍵マークの `HF_TOKEN` にアクセスを許可（これまでと同じ）。
3. 「ランタイム」→「すべてのセルを実行」。ドライブの許可を聞かれたら許可。
4. 15分ほどで、ドライブの `AKANE_アーキテクチャ/E16-0_results/E160_results.json.gz` ができます。最後のセルが速報を出します。

In [ ]:
import os, json, time, gzip, subprocess, sys
import numpy as np
from google.colab import drive, userdata
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/AKANE_アーキテクチャ/E16-0_results"
os.makedirs(OUT, exist_ok=True)
GEMMA_ID = "google/gemma-2-2b-it"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nltk"], check=False)
import nltk; nltk.download("wordnet", quiet=True)
from nltk.corpus import wordnet as wn

In [ ]:
# 中身（sims/pbit/e160/e160_core.py と同じ）
# E16-0 の中身（ノートと手元の分析で同じ物を使う）。手書きの語の一覧はない。
LET = "ABCD"


def cap(s):
    return s[:1].upper() + s[1:]


def show(ent):
    return ent.replace("_", " ")


def mc_prompt(direction, it, opts):
    body = "\n".join(f"{LET[i]}. {o}" for i, o in enumerate(opts))
    if direction == "f":
        q = f"{cap(show(it['ent']))} is a type of which of the following?"
    else:
        q = f"Which of the following is a type of {it['cname']}?"
    return f"{q}\n{body}\nAnswer with the letter only."


def mc_set(items, cats):
    """4択の問いを全部作る。正解を A〜D の4つの位置に1回ずつ置く（位置の癖を消す）。"""
    out = []
    for n, it in enumerate(items):
        for direction in "fr":
            for level in ("easy", "hard"):
                ds = it[f"{direction}_{level}"]
                if direction == "f":
                    right, wrong = cats[it["cat"]], [cats[c] for c in ds]
                else:
                    right, wrong = show(it["ent"]), [show(z) for z in ds]
                for pos in range(4):
                    opts = wrong[:pos] + [right] + wrong[pos:]
                    out.append(dict(i=n, d=direction, lv=level, pos=pos, q=mc_prompt(direction, it, opts)))
    return out


def open_f_prompt(ent):
    return f"{cap(show(ent))} is a type of what? Answer with one or two words only."


def open_r_prompt(cname, k=20):
    return f"List {k} different types of {cname}, one per line, with no other text."


def parse_list(text):
    out = []
    for ln in text.splitlines():
        s = ln.strip().lstrip("-*•0123456789.) ").strip().strip(".").lower()
        if s and s not in out:
            out.append(s)
    return out

In [ ]:
DATA = json.loads('{"note": "E16-0：一種かを前向き（物→分類）と逆向き（分類→物）の4択で聞く。WordNetの正解（E10-3 K）", "cats": {"organism.n.01": "living thing", "animal.n.01": "animal", "mammal.n.01": "mammal", "bird.n.01": "bird", "fish.n.01": "fish", "insect.n.01": "insect", "reptile.n.01": "reptile", "dog.n.01": "dog", "plant.n.02": "plant", "tree.n.01": "tree", "flower.n.01": "flower", "edible_fruit.n.01": "fruit", "vegetable.n.01": "vegetable", "food.n.01": "food", "dish.n.02": "dish", "beverage.n.01": "drink", "artifact.n.01": "man-made object", "tool.n.01": "tool", "device.n.01": "device", "machine.n.01": "machine", "equipment.n.01": "equipment", "vehicle.n.01": "vehicle", "car.n.01": "car", "boat.n.01": "boat", "aircraft.n.01": "aircraft", "weapon.n.01": "weapon", "container.n.01": "container", "furniture.n.01": "furniture", "clothing.n.01": "clothing", "footwear.n.02": "footwear", "plaything.n.01": "toy", "musical_instrument.n.01": "musical instrument", "building.n.01": "building", "room.n.01": "room", "person.n.01": "person", "worker.n.01": "worker", "artist.n.01": "artist", "athlete.n.01": "athlete", "material.n.01": "material", "fabric.n.01": "fabric", "metallic_element.n.01": "metal", "body_part.n.01": "body part", "organ.n.01": "organ", "disease.n.01": "disease", "game.n.01": "game", "sport.n.01": "sport"}, "ents": ["abolitionist", "absentee", "accelerometer", "accountant", "accused", "adjuster", "aircraft", "althea", "alumnus", "amigo", "amplifier", "anarchist", "ancestor", "anesthesiologist", "aquarium", "archbishop", "architect", "aristocrat", "astrophysicist", "atheist", "attacker", "avenger", "ax", "azalea", "baboon", "babu", "backpack", "ballerina", "banjo", "bantam", "bayonet", "beagle", "beanie", "bedroom", "beekeeper", "belting", "betel", "betrothed", "bicycle", "biennial", "bimbo", "biographer", "bistro", "boathouse", "boatman", "bobcat", "bodybuilder", "bot", "boyfriend", "brad", "bramble", "brandy", "brazier", "breeder", "briefcase", "broad", "brogan", "buggy", "bulldog", "bulldozer", "bumblebee", "burrito", "businesswoman", "buttercup", "buttermilk", "caddie", "calender", "caliper", "camellia", "candlestick", "capacitor", "capo", "carbine", "cardiologist", "caribou", "celibate", "cello", "cesspool", "chalet", "chateau", "cheetah", "chimpanzee", "chipmunk", "chisel", "choreographer", "chronicler", "chub", "churn", "claimant", "clarence", "clary", "classroom", "clinician", "clockwork", "clover", "coachman", "coauthor", "coffin", "collie", "commissar", "composer", "conch", "condor", "confidante", "conger", "contraceptive", "convalescent", "convert", "corgi", "cougar", "councilman", "countess", "coupe", "cowgirl", "cricketer", "crocodile", "crossbow", "curry", "customer", "cutthroat", "cymbal", "dachshund", "darkroom", "dauphin", "debtor", "deer", "deflector", "delinquent", "denture", "dermatologist", "desk", "dimmer", "dinky", "dirk", "dongle", "dowager", "dragoon", "dreadnought", "dredge", "dropper", "dubbing", "dug", "dynamo", "eclectic", "educator", "egalitarian", "empress", "environmentalist", "equatorial", "erica", "espresso", "ethnic", "evergreen", "exhibitor", "exporter", "extrovert", "falafel", "falcon", "fancier", "fascist", "fife", "finch", "flamingo", "flashlight", "foal", "forceps", "fuchsia", "gasket", "gat", "gavel", "geisha", "genitalia", "georgette", "girder", "gland", "glider", "godmother", "goggles", "goldfish", "grabber", "grader", "granddaughter", "grandma", "grandparent", "grandson", "greenhouse", "greyhound", "grocer", "haggis", "handcuff", "handyman", "hangman", "harmonica", "harrow", "headmaster", "heathen", "helicopter", "hillbilly", "hoarder", "homophobe", "hooch", "houseboat", "housekeeper", "housemate", "huck", "hummingbird", "husband", "hyena", "illustrator", "immigrant", "immune", "importer", "incubator", "incurable", "informer", "injector", "insider", "integrator", "interceptor", "invalid", "investor", "isolationist", "janitor", "jasmine", "jester", "juggler", "kicker", "koala", "krill", "landlord", "larva", "liberator", "librarian", "lilac", "limousine", "liza", "llama", "locksmith", "lodger", "loom", "macho", "maggot", "mahatma", "mako", "mamba", "mammal", "manger", "mangrove", "marc", "margarita", "mariposa", "married", "masjid", "mastiff", "matador", "maul", "meatball", "mensch", "mesquite", "messenger", "midshipman", "migrant", "mimic", "minion", "minivan", "minnow", "minster", "miser", "misfit", "missus", "modernist", "moll", "mongoose", "moped", "moray", "motorboat", "motorcycle", "murine", "mustang", "narc", "narrator", "national", "nawab", "necktie", "neurologist", "neutral", "newsreader", "newt", "notary", "oboe", "ombudsman", "overlord", "paella", "pagoda", "paintbrush", "paisley", "palmyra", "panzer", "papillon", "paramedic", "paranoid", "pasha", "patriot", "pedophile", "pelican", "penis", "peony", "peregrine", "perfectionist", "peripheral", "philanthropist", "physicist", "physiologist", "piccolo", "pickpocket", "piglet", "pinto", "playmaker", "playmate", "pleaser", "plenipotentiary", "plow", "plumber", "plush", "poi", "poncho", "pontifical", "portable", "potion", "preteen", "princess", "probable", "progenitor", "programmer", "protectionist", "prowler", "psychiatrist", "psychotherapist", "pug", "pulley", "puppeteer", "quitter", "racist", "radar", "radiologist", "raglan", "rapier", "ratchet", "rayon", "recliner", "redneck", "retailer", "revisionist", "rhinoceros", "ridley", "rifle", "rival", "rodent", "rogue", "rowan", "rustic", "sabra", "salesperson", "sandwich", "sarcophagus", "sarin", "sauerkraut", "scalpel", "scapegoat", "scion", "scraper", "scythe", "secessionist", "sectional", "semaphore", "senator", "senna", "shaheed", "shed", "sheeting", "sherry", "shim", "ship", "shirt", "shiv", "shogun", "sickle", "sieve", "skiff", "sled", "slob", "snoop", "soccer", "socialist", "socialite", "sofa", "spandex", "spastic", "speakeasy", "spitfire", "spokeswoman", "sprinkler", "sprinter", "stagecoach", "stapler", "starfish", "starling", "starship", "statesman", "steamboat", "steeper", "stepdaughter", "stepfather", "stepmother", "stingray", "stockman", "stoner", "storeroom", "stork", "stray", "succulent", "suiting", "sunflower", "supermodel", "supremacist", "surrealist", "survivalist", "swami", "swine", "swot", "syringe", "tabor", "tailor", "takin", "tambourine", "tartan", "tavern", "teapot", "telegraph", "telescope", "therapist", "thermos", "thwart", "tights", "toby", "toddler", "tongs", "tortoise", "totalitarian", "towel", "transponder", "treasurer", "tricycle", "trident", "tugboat", "tun", "turbine", "tweeter", "undercarriage", "underdog", "undergraduate", "undersecretary", "understudy", "undies", "untouchable", "urchin", "valedictorian", "valerian", "valet", "vaporizer", "vermouth", "veronica", "vise", "visitor", "voyeur", "waitress", "walrus", "wanton", "warden", "warlord", "warrior", "washroom", "watchman", "waveguide", "weatherman", "welder", "wheelchair", "whiskey", "wife", "wildflower", "windowsill", "wombat", "woodpecker", "workstation", "worthy", "wraparound", "wrestler", "wristwatch", "yacht", "yucca", "zebra"], "items": [{"ent": "abolitionist", "cat": "person.n.01", "cname": "person", "f_easy": ["animal.n.01", "plaything.n.01", "food.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "mammal.n.01"], "f_hard_n": 2, "r_easy": ["llama", "waveguide", "pug"], "r_hard": ["conch", "clary", "radar"], "r_hard_n": 2}, {"ent": "absentee", "cat": "person.n.01", "cname": "person", "f_easy": ["vegetable.n.01", "plaything.n.01", "musical_instrument.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "game.n.01"], "f_hard_n": 2, "r_easy": ["conger", "sandwich", "buggy"], "r_hard": ["lilac", "buttercup", "shiv"], "r_hard_n": 2}, {"ent": "accelerometer", "cat": "device.n.01", "cname": "device", "f_easy": ["clothing.n.01", "furniture.n.01", "vegetable.n.01"], "f_hard": ["furniture.n.01", "equipment.n.01", "tool.n.01"], "f_hard_n": 3, "r_easy": ["tugboat", "biographer", "grader"], "r_hard": ["tavern", "sickle", "amplifier"], "r_hard_n": 3}, {"ent": "accountant", "cat": "person.n.01", "cname": "person", "f_easy": ["vegetable.n.01", "sport.n.01", "plaything.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "reptile.n.01"], "f_hard_n": 2, "r_easy": ["dinky", "syringe", "wristwatch"], "r_hard": ["bot", "senna", "mammal"], "r_hard_n": 2}, {"ent": "accused", "cat": "person.n.01", "cname": "person", "f_easy": ["flower.n.01", "mammal.n.01", "plant.n.02"], "f_hard": ["plant.n.02", "animal.n.01", "edible_fruit.n.01"], "f_hard_n": 2, "r_easy": ["whiskey", "motorcycle", "tambourine"], "r_hard": ["starfish", "wildflower", "classroom"], "r_hard_n": 2}, {"ent": "adjuster", "cat": "person.n.01", "cname": "person", "f_easy": ["disease.n.01", "edible_fruit.n.01", "reptile.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "reptile.n.01"], "f_hard_n": 2, "r_easy": ["minivan", "calender", "accelerometer"], "r_hard": ["starfish", "newt", "dug"], "r_hard_n": 2}, {"ent": "aircraft", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["dog.n.01", "bird.n.01", "fish.n.01"], "f_hard": ["tool.n.01", "equipment.n.01", "building.n.01"], "f_hard_n": 3, "r_easy": ["rifle", "neutral", "peripheral"], "r_hard": ["dongle", "bistro", "equatorial"], "r_hard_n": 3}, {"ent": "althea", "cat": "plant.n.02", "cname": "plant", "f_easy": ["equipment.n.01", "worker.n.01", "reptile.n.01"], "f_hard": ["animal.n.01", "person.n.01", "worker.n.01"], "f_hard_n": 2, "r_easy": ["informer", "dragoon", "bulldog"], "r_hard": ["newsreader", "mahatma", "ancestor"], "r_hard_n": 2}, {"ent": "alumnus", "cat": "person.n.01", "cname": "person", "f_easy": ["aircraft.n.01", "footwear.n.02", "vegetable.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "bird.n.01"], "f_hard_n": 2, "r_easy": ["wildflower", "syringe", "shiv"], "r_hard": ["maggot", "piglet", "masjid"], "r_hard_n": 2}, {"ent": "amigo", "cat": "person.n.01", "cname": "person", "f_easy": ["material.n.01", "musical_instrument.n.01", "insect.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "dog.n.01"], "f_hard_n": 2, "r_easy": ["wraparound", "paisley", "manger"], "r_hard": ["yucca", "mariposa", "mako"], "r_hard_n": 2}, {"ent": "amplifier", "cat": "equipment.n.01", "cname": "equipment", "f_easy": ["footwear.n.02", "dish.n.02", "clothing.n.01"], "f_hard": ["plaything.n.01", "device.n.01", "tool.n.01"], "f_hard_n": 3, "r_easy": ["immune", "huck", "potion"], "r_hard": ["thermos", "sarcophagus", "sieve"], "r_hard_n": 3}, {"ent": "anarchist", "cat": "person.n.01", "cname": "person", "f_easy": ["edible_fruit.n.01", "reptile.n.01", "sport.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "disease.n.01"], "f_hard_n": 2, "r_easy": ["cello", "pulley", "mongoose"], "r_hard": ["camellia", "mesquite", "aquarium"], "r_hard_n": 2}, {"ent": "ancestor", "cat": "person.n.01", "cname": "person", "f_easy": ["material.n.01", "insect.n.01", "plant.n.02"], "f_hard": ["plant.n.02", "animal.n.01", "boat.n.01"], "f_hard_n": 2, "r_easy": ["houseboat", "spandex", "espresso"], "r_hard": ["wildflower", "foal", "windowsill"], "r_hard_n": 2}, {"ent": "anesthesiologist", "cat": "person.n.01", "cname": "person", "f_easy": ["device.n.01", "dog.n.01", "sport.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "mammal.n.01"], "f_hard_n": 2, "r_easy": ["minnow", "sprinkler", "raglan"], "r_hard": ["althea", "starfish", "undercarriage"], "r_hard_n": 2}, {"ent": "aquarium", "cat": "container.n.01", "cname": "container", "f_easy": ["vehicle.n.01", "bird.n.01", "equipment.n.01"], "f_hard": ["fabric.n.01", "furniture.n.01", "building.n.01"], "f_hard_n": 3, "r_easy": ["beanie", "llama", "tights"], "r_hard": ["scythe", "chalet", "clockwork"], "r_hard_n": 3}, {"ent": "archbishop", "cat": "person.n.01", "cname": "person", "f_easy": ["clothing.n.01", "boat.n.01", "insect.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "beverage.n.01"], "f_hard_n": 2, "r_easy": ["scythe", "shirt", "sunflower"], "r_hard": ["krill", "mariposa", "betel"], "r_hard_n": 2}, {"ent": "architect", "cat": "person.n.01", "cname": "person", "f_easy": ["fish.n.01", "dog.n.01", "tool.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "fish.n.01"], "f_hard_n": 2, "r_easy": ["corgi", "harrow", "undercarriage"], "r_hard": ["mesquite", "newt", "zebra"], "r_hard_n": 2}, {"ent": "aristocrat", "cat": "person.n.01", "cname": "person", "f_easy": ["fabric.n.01", "room.n.01", "dog.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "edible_fruit.n.01"], "f_hard_n": 2, "r_easy": ["conch", "tights", "dreadnought"], "r_hard": ["newt", "mammal", "bantam"], "r_hard_n": 2}, {"ent": "astrophysicist", "cat": "person.n.01", "cname": "person", "f_easy": ["dish.n.02", "tool.n.01", "beverage.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "footwear.n.02"], "f_hard_n": 2, "r_easy": ["gavel", "mamba", "buttercup"], "r_hard": ["foal", "succulent", "tortoise"], "r_hard_n": 2}, {"ent": "atheist", "cat": "person.n.01", "cname": "person", "f_easy": ["sport.n.01", "plaything.n.01", "room.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "game.n.01"], "f_hard_n": 2, "r_easy": ["penis", "motorcycle", "larva"], "r_hard": ["succulent", "biennial", "potion"], "r_hard_n": 2}, {"ent": "attacker", "cat": "person.n.01", "cname": "person", "f_easy": ["dish.n.02", "car.n.01", "organ.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "sport.n.01"], "f_hard_n": 2, "r_easy": ["betel", "oboe", "dinky"], "r_hard": ["mariposa", "krill", "bedroom"], "r_hard_n": 2}, {"ent": "avenger", "cat": "person.n.01", "cname": "person", "f_easy": ["furniture.n.01", "equipment.n.01", "beverage.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "building.n.01"], "f_hard_n": 2, "r_easy": ["genitalia", "speakeasy", "stapler"], "r_hard": ["yucca", "foal", "haggis"], "r_hard_n": 2}, {"ent": "ax", "cat": "tool.n.01", "cname": "tool", "f_easy": ["fish.n.01", "metallic_element.n.01", "organ.n.01"], "f_hard": ["plaything.n.01", "device.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["biennial", "collie", "falcon"], "r_hard": ["chateau", "girder", "suiting"], "r_hard_n": 3}, {"ent": "azalea", "cat": "plant.n.02", "cname": "plant", "f_easy": ["container.n.01", "equipment.n.01", "organ.n.01"], "f_hard": ["animal.n.01", "person.n.01", "weapon.n.01"], "f_hard_n": 2, "r_easy": ["hooch", "goggles", "supremacist"], "r_hard": ["physiologist", "macho", "stork"], "r_hard_n": 2}, {"ent": "baboon", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["body_part.n.01", "flower.n.01", "person.n.01"], "f_hard": ["fish.n.01", "reptile.n.01", "insect.n.01"], "f_hard_n": 3, "r_easy": ["masjid", "narrator", "debtor"], "r_hard": ["hummingbird", "pelican", "chub"], "r_hard_n": 3}, {"ent": "babu", "cat": "person.n.01", "cname": "person", "f_easy": ["body_part.n.01", "tool.n.01", "artifact.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "dish.n.02"], "f_hard_n": 2, "r_easy": ["cymbal", "burrito", "radar"], "r_hard": ["starfish", "piglet", "rhinoceros"], "r_hard_n": 2}, {"ent": "backpack", "cat": "container.n.01", "cname": "container", "f_easy": ["boat.n.01", "game.n.01", "weapon.n.01"], "f_hard": ["tool.n.01", "plaything.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["swot", "sabra", "messenger"], "r_hard": ["tartan", "tongs", "brogan"], "r_hard_n": 3}, {"ent": "ballerina", "cat": "person.n.01", "cname": "person", "f_easy": ["insect.n.01", "sport.n.01", "vehicle.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "material.n.01"], "f_hard_n": 2, "r_easy": ["helicopter", "tun", "hyena"], "r_hard": ["maggot", "clover", "larva"], "r_hard_n": 2}, {"ent": "banjo", "cat": "musical_instrument.n.01", "cname": "musical instrument", "f_easy": ["plaything.n.01", "dish.n.02", "furniture.n.01"], "f_hard": ["machine.n.01", "weapon.n.01", "fabric.n.01"], "f_hard_n": 2, "r_easy": ["sauerkraut", "stepfather", "rival"], "r_hard": ["rifle", "stapler", "telegraph"], "r_hard_n": 2}, {"ent": "bantam", "cat": "bird.n.01", "cname": "bird", "f_easy": ["edible_fruit.n.01", "athlete.n.01", "furniture.n.01"], "f_hard": ["insect.n.01", "reptile.n.01", "fish.n.01"], "f_hard_n": 3, "r_easy": ["workstation", "collie", "plumber"], "r_hard": ["swine", "bobcat", "bumblebee"], "r_hard_n": 3}, {"ent": "bayonet", "cat": "weapon.n.01", "cname": "weapon", "f_easy": ["sport.n.01", "beverage.n.01", "food.n.01"], "f_hard": ["musical_instrument.n.01", "machine.n.01", "organism.n.01"], "f_hard_n": 2, "r_easy": ["survivalist", "undersecretary", "playmate"], "r_hard": ["turbine", "dredge", "denture"], "r_hard_n": 2}, {"ent": "beagle", "cat": "dog.n.01", "cname": "dog", "f_easy": ["aircraft.n.01", "reptile.n.01", "edible_fruit.n.01"], "f_hard": ["fabric.n.01", "reptile.n.01", "artifact.n.01"], "f_hard_n": 0, "r_easy": ["quitter", "spokeswoman", "panzer"], "r_hard": ["princess", "haggis", "dinky"], "r_hard_n": 0}, {"ent": "beanie", "cat": "clothing.n.01", "cname": "clothing", "f_easy": ["food.n.01", "fish.n.01", "plaything.n.01"], "f_hard": ["building.n.01", "room.n.01", "furniture.n.01"], "f_hard_n": 3, "r_easy": ["sarin", "integrator", "thermos"], "r_hard": ["girder", "deflector", "ax"], "r_hard_n": 3}, {"ent": "bedroom", "cat": "room.n.01", "cname": "room", "f_easy": ["footwear.n.02", "vehicle.n.01", "organ.n.01"], "f_hard": ["device.n.01", "plaything.n.01", "clothing.n.01"], "f_hard_n": 3, "r_easy": ["shiv", "stepmother", "stagecoach"], "r_hard": ["ship", "ax", "steeper"], "r_hard_n": 3}, {"ent": "beekeeper", "cat": "person.n.01", "cname": "person", "f_easy": ["food.n.01", "machine.n.01", "furniture.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "car.n.01"], "f_hard_n": 2, "r_easy": ["cymbal", "falcon", "yacht"], "r_hard": ["starfish", "senna", "rifle"], "r_hard_n": 2}, {"ent": "belting", "cat": "fabric.n.01", "cname": "fabric", "f_easy": ["aircraft.n.01", "plant.n.02", "tool.n.01"], "f_hard": ["vehicle.n.01", "device.n.01", "tool.n.01"], "f_hard_n": 3, "r_easy": ["waveguide", "shogun", "girder"], "r_hard": ["necktie", "clarence", "forceps"], "r_hard_n": 3}, {"ent": "betel", "cat": "plant.n.02", "cname": "plant", "f_easy": ["person.n.01", "sport.n.01", "vegetable.n.01"], "f_hard": ["person.n.01", "animal.n.01", "edible_fruit.n.01"], "f_hard_n": 2, "r_easy": ["tugboat", "midshipman", "immigrant"], "r_hard": ["socialist", "immune", "teapot"], "r_hard_n": 2}, {"ent": "betrothed", "cat": "person.n.01", "cname": "person", "f_easy": ["edible_fruit.n.01", "beverage.n.01", "building.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "body_part.n.01"], "f_hard_n": 2, "r_easy": ["biennial", "palmyra", "tun"], "r_hard": ["starfish", "evergreen", "semaphore"], "r_hard_n": 2}, {"ent": "bicycle", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["mammal.n.01", "reptile.n.01", "fabric.n.01"], "f_hard": ["building.n.01", "room.n.01", "clothing.n.01"], "f_hard_n": 3, "r_easy": ["koala", "misfit", "coauthor"], "r_hard": ["tavern", "injector", "brogan"], "r_hard_n": 3}, {"ent": "bicycle", "cat": "container.n.01", "cname": "container", "f_easy": ["worker.n.01", "fish.n.01", "room.n.01"], "f_hard": ["building.n.01", "fabric.n.01", "furniture.n.01"], "f_hard_n": 3, "r_easy": ["misfit", "walrus", "nawab"], "r_hard": ["necktie", "tartan", "georgette"], "r_hard_n": 3}, {"ent": "biennial", "cat": "plant.n.02", "cname": "plant", "f_easy": ["game.n.01", "disease.n.01", "machine.n.01"], "f_hard": ["person.n.01", "animal.n.01", "vegetable.n.01"], "f_hard_n": 2, "r_easy": ["physicist", "perfectionist", "locksmith"], "r_hard": ["tailor", "businesswoman", "houseboat"], "r_hard_n": 2}, {"ent": "bimbo", "cat": "person.n.01", "cname": "person", "f_easy": ["musical_instrument.n.01", "boat.n.01", "sport.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "insect.n.01"], "f_hard_n": 2, "r_easy": ["greyhound", "bicycle", "fife"], "r_hard": ["conch", "erica", "mustang"], "r_hard_n": 2}, {"ent": "biographer", "cat": "person.n.01", "cname": "person", "f_easy": ["bird.n.01", "equipment.n.01", "vehicle.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "vegetable.n.01"], "f_hard_n": 2, "r_easy": ["boathouse", "vise", "azalea"], "r_hard": ["succulent", "mesquite", "aircraft"], "r_hard_n": 2}, {"ent": "bistro", "cat": "building.n.01", "cname": "building", "f_easy": ["room.n.01", "tool.n.01", "machine.n.01"], "f_hard": ["plaything.n.01", "footwear.n.02", "container.n.01"], "f_hard_n": 3, "r_easy": ["egalitarian", "recliner", "loom"], "r_hard": ["toby", "plush", "buggy"], "r_hard_n": 3}, {"ent": "boathouse", "cat": "building.n.01", "cname": "building", "f_easy": ["room.n.01", "worker.n.01", "plaything.n.01"], "f_hard": ["container.n.01", "fabric.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["eclectic", "piglet", "briefcase"], "r_hard": ["gavel", "dubbing", "desk"], "r_hard_n": 3}, {"ent": "boatman", "cat": "worker.n.01", "cname": "worker", "f_easy": ["body_part.n.01", "mammal.n.01", "fish.n.01"], "f_hard": ["artist.n.01", "athlete.n.01", "flower.n.01"], "f_hard_n": 2, "r_easy": ["potion", "claimant", "warden"], "r_hard": ["kicker", "sprinter", "gavel"], "r_hard_n": 2}, {"ent": "bobcat", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["device.n.01", "reptile.n.01", "athlete.n.01"], "f_hard": ["bird.n.01", "fish.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["condor", "paranoid", "weatherman"], "r_hard": ["woodpecker", "bumblebee", "stork"], "r_hard_n": 3}, {"ent": "bodybuilder", "cat": "person.n.01", "cname": "person", "f_easy": ["plaything.n.01", "material.n.01", "plant.n.02"], "f_hard": ["animal.n.01", "plant.n.02", "equipment.n.01"], "f_hard_n": 2, "r_easy": ["paella", "hyena", "clarence"], "r_hard": ["piglet", "yucca", "betel"], "r_hard_n": 2}, {"ent": "bot", "cat": "animal.n.01", "cname": "animal", "f_easy": ["artifact.n.01", "equipment.n.01", "artist.n.01"], "f_hard": ["plant.n.02", "person.n.01", "boat.n.01"], "f_hard_n": 2, "r_easy": ["forceps", "ballerina", "programmer"], "r_hard": ["anarchist", "statesman", "greenhouse"], "r_hard_n": 2}, {"ent": "boyfriend", "cat": "person.n.01", "cname": "person", "f_easy": ["vehicle.n.01", "plaything.n.01", "footwear.n.02"], "f_hard": ["animal.n.01", "plant.n.02", "insect.n.01"], "f_hard_n": 2, "r_easy": ["krill", "mustang", "vaporizer"], "r_hard": ["foal", "larva", "peony"], "r_hard_n": 2}, {"ent": "brad", "cat": "device.n.01", "cname": "device", "f_easy": ["person.n.01", "furniture.n.01", "edible_fruit.n.01"], "f_hard": ["tool.n.01", "building.n.01", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["countess", "marc", "warlord"], "r_hard": ["masjid", "chisel", "briefcase"], "r_hard_n": 3}, {"ent": "bramble", "cat": "plant.n.02", "cname": "plant", "f_easy": ["boat.n.01", "mammal.n.01", "body_part.n.01"], "f_hard": ["animal.n.01", "person.n.01", "reptile.n.01"], "f_hard_n": 2, "r_easy": ["librarian", "dimmer", "boathouse"], "r_hard": ["clinician", "patriot", "shaheed"], "r_hard_n": 2}, {"ent": "brandy", "cat": "beverage.n.01", "cname": "drink", "f_easy": ["plant.n.02", "disease.n.01", "athlete.n.01"], "f_hard": ["dish.n.02", "building.n.01", "tree.n.01"], "f_hard_n": 1, "r_easy": ["lodger", "isolationist", "pontifical"], "r_hard": ["falafel", "pasha", "bodybuilder"], "r_hard_n": 1}, {"ent": "brazier", "cat": "device.n.01", "cname": "device", "f_easy": ["organ.n.01", "tree.n.01", "animal.n.01"], "f_hard": ["clothing.n.01", "footwear.n.02", "building.n.01"], "f_hard_n": 3, "r_easy": ["mastiff", "ballerina", "revisionist"], "r_hard": ["manger", "bistro", "sickle"], "r_hard_n": 3}, {"ent": "breeder", "cat": "person.n.01", "cname": "person", "f_easy": ["machine.n.01", "tree.n.01", "insect.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "reptile.n.01"], "f_hard_n": 2, "r_easy": ["tweeter", "betel", "poncho"], "r_hard": ["newt", "piglet", "scalpel"], "r_hard_n": 2}, {"ent": "briefcase", "cat": "container.n.01", "cname": "container", "f_easy": ["athlete.n.01", "body_part.n.01", "clothing.n.01"], "f_hard": ["footwear.n.02", "fabric.n.01", "device.n.01"], "f_hard_n": 3, "r_easy": ["shogun", "hyena", "missus"], "r_hard": ["plow", "masjid", "equatorial"], "r_hard_n": 3}, {"ent": "broad", "cat": "person.n.01", "cname": "person", "f_easy": ["container.n.01", "plant.n.02", "fabric.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "tool.n.01"], "f_hard_n": 2, "r_easy": ["calender", "chipmunk", "rayon"], "r_hard": ["krill", "fuchsia", "sunflower"], "r_hard_n": 2}, {"ent": "brogan", "cat": "footwear.n.02", "cname": "footwear", "f_easy": ["room.n.01", "athlete.n.01", "tool.n.01"], "f_hard": ["clothing.n.01", "vehicle.n.01", "container.n.01"], "f_hard_n": 3, "r_easy": ["vaporizer", "architect", "liberator"], "r_hard": ["pulley", "greenhouse", "poncho"], "r_hard_n": 3}, {"ent": "buggy", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["body_part.n.01", "artist.n.01", "furniture.n.01"], "f_hard": ["building.n.01", "clothing.n.01", "room.n.01"], "f_hard_n": 3, "r_easy": ["godmother", "babu", "swine"], "r_hard": ["plush", "vaporizer", "raglan"], "r_hard_n": 3}, {"ent": "buggy", "cat": "container.n.01", "cname": "container", "f_easy": ["bird.n.01", "beverage.n.01", "body_part.n.01"], "f_hard": ["room.n.01", "plaything.n.01", "building.n.01"], "f_hard_n": 3, "r_easy": ["playmate", "exhibitor", "speakeasy"], "r_hard": ["chateau", "scalpel", "dimmer"], "r_hard_n": 3}, {"ent": "bulldog", "cat": "dog.n.01", "cname": "dog", "f_easy": ["weapon.n.01", "device.n.01", "insect.n.01"], "f_hard": ["plaything.n.01", "machine.n.01", "vehicle.n.01"], "f_hard_n": 0, "r_easy": ["jasmine", "accountant", "grandma"], "r_hard": ["boyfriend", "tabor", "dowager"], "r_hard_n": 0}, {"ent": "bulldozer", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["disease.n.01", "beverage.n.01", "mammal.n.01"], "f_hard": ["room.n.01", "building.n.01", "tool.n.01"], "f_hard_n": 3, "r_easy": ["psychotherapist", "salesperson", "wombat"], "r_hard": ["necktie", "windowsill", "churn"], "r_hard_n": 3}, {"ent": "bulldozer", "cat": "container.n.01", "cname": "container", "f_easy": ["musical_instrument.n.01", "vegetable.n.01", "beverage.n.01"], "f_hard": ["tool.n.01", "furniture.n.01", "fabric.n.01"], "f_hard_n": 3, "r_easy": ["warden", "ratchet", "tambourine"], "r_hard": ["scythe", "yacht", "transponder"], "r_hard_n": 3}, {"ent": "bumblebee", "cat": "insect.n.01", "cname": "insect", "f_easy": ["person.n.01", "footwear.n.02", "equipment.n.01"], "f_hard": ["fish.n.01", "mammal.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["sheeting", "azalea", "meatball"], "r_hard": ["chipmunk", "stingray", "baboon"], "r_hard_n": 3}, {"ent": "burrito", "cat": "dish.n.02", "cname": "dish", "f_easy": ["reptile.n.01", "equipment.n.01", "container.n.01"], "f_hard": ["beverage.n.01", "plant.n.02", "game.n.01"], "f_hard_n": 1, "r_easy": ["newsreader", "pagoda", "hummingbird"], "r_hard": ["whiskey", "bayonet", "georgette"], "r_hard_n": 1}, {"ent": "businesswoman", "cat": "person.n.01", "cname": "person", "f_easy": ["aircraft.n.01", "insect.n.01", "tree.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "disease.n.01"], "f_hard_n": 2, "r_easy": ["huck", "tabor", "masjid"], "r_hard": ["yucca", "succulent", "maggot"], "r_hard_n": 2}, {"ent": "buttercup", "cat": "plant.n.02", "cname": "plant", "f_easy": ["disease.n.01", "device.n.01", "dog.n.01"], "f_hard": ["animal.n.01", "person.n.01", "material.n.01"], "f_hard_n": 2, "r_easy": ["wrestler", "carbine", "spandex"], "r_hard": ["warrior", "educator", "motorboat"], "r_hard_n": 2}, {"ent": "buttermilk", "cat": "beverage.n.01", "cname": "drink", "f_easy": ["game.n.01", "artist.n.01", "edible_fruit.n.01"], "f_hard": ["dish.n.02", "vegetable.n.01", "clothing.n.01"], "f_hard_n": 1, "r_easy": ["rogue", "eclectic", "rayon"], "r_hard": ["sandwich", "tongs", "miser"], "r_hard_n": 1}, {"ent": "caddie", "cat": "worker.n.01", "cname": "worker", "f_easy": ["musical_instrument.n.01", "dish.n.02", "aircraft.n.01"], "f_hard": ["artist.n.01", "athlete.n.01", "disease.n.01"], "f_hard_n": 2, "r_easy": ["peripheral", "chronicler", "shaheed"], "r_hard": ["illustrator", "sprinter", "shed"], "r_hard_n": 2}, {"ent": "calender", "cat": "machine.n.01", "cname": "machine", "f_easy": ["dish.n.02", "worker.n.01", "fish.n.01"], "f_hard": ["weapon.n.01", "musical_instrument.n.01", "building.n.01"], "f_hard_n": 2, "r_easy": ["handyman", "trident", "sprinter"], "r_hard": ["trident", "rifle", "starling"], "r_hard_n": 2}, {"ent": "caliper", "cat": "device.n.01", "cname": "device", "f_easy": ["plaything.n.01", "dog.n.01", "dish.n.02"], "f_hard": ["clothing.n.01", "equipment.n.01", "furniture.n.01"], "f_hard_n": 3, "r_easy": ["anarchist", "claimant", "steeper"], "r_hard": ["paisley", "bulldozer", "wheelchair"], "r_hard_n": 3}, {"ent": "camellia", "cat": "plant.n.02", "cname": "plant", "f_easy": ["artist.n.01", "container.n.01", "beverage.n.01"], "f_hard": ["person.n.01", "animal.n.01", "worker.n.01"], "f_hard_n": 2, "r_easy": ["tavern", "tweeter", "eclectic"], "r_hard": ["spitfire", "paranoid", "convalescent"], "r_hard_n": 2}, {"ent": "candlestick", "cat": "device.n.01", "cname": "device", "f_easy": ["furniture.n.01", "plant.n.02", "game.n.01"], "f_hard": ["room.n.01", "fabric.n.01", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["breeder", "warden", "patriot"], "r_hard": ["storeroom", "tun", "coffin"], "r_hard_n": 3}, {"ent": "capacitor", "cat": "device.n.01", "cname": "device", "f_easy": ["insect.n.01", "beverage.n.01", "vehicle.n.01"], "f_hard": ["container.n.01", "clothing.n.01", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["undersecretary", "undies", "husband"], "r_hard": ["bedroom", "clarence", "stagecoach"], "r_hard_n": 3}, {"ent": "capo", "cat": "person.n.01", "cname": "person", "f_easy": ["metallic_element.n.01", "boat.n.01", "bird.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "fabric.n.01"], "f_hard_n": 2, "r_easy": ["pontifical", "tortoise", "workstation"], "r_hard": ["betel", "mammal", "soccer"], "r_hard_n": 2}, {"ent": "carbine", "cat": "weapon.n.01", "cname": "weapon", "f_easy": ["clothing.n.01", "worker.n.01", "footwear.n.02"], "f_hard": ["musical_instrument.n.01", "machine.n.01", "fish.n.01"], "f_hard_n": 2, "r_easy": ["scion", "rival", "grocer"], "r_hard": ["stapler", "workstation", "migrant"], "r_hard_n": 2}, {"ent": "cardiologist", "cat": "person.n.01", "cname": "person", "f_easy": ["dish.n.02", "game.n.01", "building.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "bird.n.01"], "f_hard_n": 2, "r_easy": ["mangrove", "backpack", "fuchsia"], "r_hard": ["senna", "succulent", "espresso"], "r_hard_n": 2}, {"ent": "caribou", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["plaything.n.01", "sport.n.01", "room.n.01"], "f_hard": ["insect.n.01", "fish.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["grandson", "valedictorian", "camellia"], "r_hard": ["starling", "condor", "peregrine"], "r_hard_n": 3}, {"ent": "celibate", "cat": "person.n.01", "cname": "person", "f_easy": ["tool.n.01", "container.n.01", "weapon.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "vegetable.n.01"], "f_hard_n": 2, "r_easy": ["helicopter", "aquarium", "gasket"], "r_hard": ["stray", "biennial", "bedroom"], "r_hard_n": 2}, {"ent": "cello", "cat": "musical_instrument.n.01", "cname": "musical instrument", "f_easy": ["person.n.01", "flower.n.01", "equipment.n.01"], "f_hard": ["weapon.n.01", "machine.n.01", "aircraft.n.01"], "f_hard_n": 2, "r_easy": ["rayon", "azalea", "bantam"], "r_hard": ["bayonet", "turbine", "fascist"], "r_hard_n": 2}, {"ent": "cesspool", "cat": "container.n.01", "cname": "container", "f_easy": ["insect.n.01", "bird.n.01", "footwear.n.02"], "f_hard": ["footwear.n.02", "furniture.n.01", "vehicle.n.01"], "f_hard_n": 3, "r_easy": ["paisley", "worthy", "misfit"], "r_hard": ["bedroom", "scraper", "chalet"], "r_hard_n": 3}, {"ent": "chalet", "cat": "building.n.01", "cname": "building", "f_easy": ["bird.n.01", "body_part.n.01", "container.n.01"], "f_hard": ["furniture.n.01", "tool.n.01", "fabric.n.01"], "f_hard_n": 3, "r_easy": ["classroom", "mesquite", "equatorial"], "r_hard": ["transponder", "sectional", "plow"], "r_hard_n": 3}, {"ent": "chateau", "cat": "building.n.01", "cname": "building", "f_easy": ["person.n.01", "device.n.01", "vegetable.n.01"], "f_hard": ["furniture.n.01", "device.n.01", "tool.n.01"], "f_hard_n": 3, "r_easy": ["liza", "survivalist", "confidante"], "r_hard": ["vaporizer", "desk", "candlestick"], "r_hard_n": 3}, {"ent": "cheetah", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["body_part.n.01", "boat.n.01", "machine.n.01"], "f_hard": ["bird.n.01", "insect.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["undercarriage", "senator", "absentee"], "r_hard": ["stingray", "stork", "mako"], "r_hard_n": 3}, {"ent": "chimpanzee", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["artifact.n.01", "vehicle.n.01", "tool.n.01"], "f_hard": ["insect.n.01", "fish.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["caddie", "babu", "midshipman"], "r_hard": ["starling", "liza", "flamingo"], "r_hard_n": 3}, {"ent": "chipmunk", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["weapon.n.01", "person.n.01", "vegetable.n.01"], "f_hard": ["reptile.n.01", "insect.n.01", "bird.n.01"], "f_hard_n": 3, "r_easy": ["sprinter", "paranoid", "grabber"], "r_hard": ["mako", "ridley", "stork"], "r_hard_n": 3}, {"ent": "chisel", "cat": "tool.n.01", "cname": "tool", "f_easy": ["musical_instrument.n.01", "metallic_element.n.01", "footwear.n.02"], "f_hard": ["vehicle.n.01", "room.n.01", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["stockman", "chateau", "notary"], "r_hard": ["coffin", "peripheral", "pulley"], "r_hard_n": 3}, {"ent": "choreographer", "cat": "person.n.01", "cname": "person", "f_easy": ["dish.n.02", "vegetable.n.01", "tree.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "footwear.n.02"], "f_hard_n": 2, "r_easy": ["accelerometer", "chalet", "brad"], "r_hard": ["conch", "stray", "gavel"], "r_hard_n": 2}, {"ent": "chronicler", "cat": "person.n.01", "cname": "person", "f_easy": ["boat.n.01", "game.n.01", "body_part.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "device.n.01"], "f_hard_n": 2, "r_easy": ["belting", "dredge", "penis"], "r_hard": ["mammal", "mariposa", "deflector"], "r_hard_n": 2}, {"ent": "chub", "cat": "fish.n.01", "cname": "fish", "f_easy": ["reptile.n.01", "bird.n.01", "beverage.n.01"], "f_hard": ["reptile.n.01", "insect.n.01", "mammal.n.01"], "f_hard_n": 3, "r_easy": ["motorboat", "desk", "flashlight"], "r_hard": ["zebra", "koala", "condor"], "r_hard_n": 3}, {"ent": "churn", "cat": "container.n.01", "cname": "container", "f_easy": ["worker.n.01", "mammal.n.01", "athlete.n.01"], "f_hard": ["device.n.01", "clothing.n.01", "vehicle.n.01"], "f_hard_n": 3, "r_easy": ["bumblebee", "bistro", "syringe"], "r_hard": ["brogan", "clockwork", "towel"], "r_hard_n": 3}, {"ent": "claimant", "cat": "person.n.01", "cname": "person", "f_easy": ["furniture.n.01", "container.n.01", "boat.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "container.n.01"], "f_hard_n": 2, "r_easy": ["vise", "shiv", "brandy"], "r_hard": ["larva", "stray", "bot"], "r_hard_n": 2}, {"ent": "clarence", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["animal.n.01", "worker.n.01", "edible_fruit.n.01"], "f_hard": ["footwear.n.02", "room.n.01", "building.n.01"], "f_hard_n": 3, "r_easy": ["cricketer", "socialite", "corgi"], "r_hard": ["deflector", "wraparound", "wristwatch"], "r_hard_n": 3}, {"ent": "clarence", "cat": "container.n.01", "cname": "container", "f_easy": ["plaything.n.01", "worker.n.01", "disease.n.01"], "f_hard": ["building.n.01", "device.n.01", "furniture.n.01"], "f_hard_n": 3, "r_easy": ["librarian", "married", "poi"], "r_hard": ["scalpel", "peripheral", "equatorial"], "r_hard_n": 3}, {"ent": "clary", "cat": "plant.n.02", "cname": "plant", "f_easy": ["worker.n.01", "plaything.n.01", "person.n.01"], "f_hard": ["animal.n.01", "person.n.01", "boat.n.01"], "f_hard_n": 2, "r_easy": ["supermodel", "piccolo", "pontifical"], "r_hard": ["extrovert", "importer", "wrestler"], "r_hard_n": 2}, {"ent": "classroom", "cat": "room.n.01", "cname": "room", "f_easy": ["material.n.01", "organism.n.01", "vehicle.n.01"], "f_hard": ["building.n.01", "device.n.01", "clothing.n.01"], "f_hard_n": 3, "r_easy": ["aircraft", "coupe", "bayonet"], "r_hard": ["accelerometer", "pontifical", "toby"], "r_hard_n": 3}, {"ent": "clinician", "cat": "person.n.01", "cname": "person", "f_easy": ["organ.n.01", "insect.n.01", "bird.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "machine.n.01"], "f_hard_n": 2, "r_easy": ["minster", "telescope", "baboon"], "r_hard": ["clary", "krill", "flamingo"], "r_hard_n": 2}, {"ent": "clockwork", "cat": "device.n.01", "cname": "device", "f_easy": ["edible_fruit.n.01", "worker.n.01", "disease.n.01"], "f_hard": ["container.n.01", "furniture.n.01", "tool.n.01"], "f_hard_n": 3, "r_easy": ["broad", "starling", "protectionist"], "r_hard": ["belting", "toby", "sarcophagus"], "r_hard_n": 3}, {"ent": "clover", "cat": "plant.n.02", "cname": "plant", "f_easy": ["edible_fruit.n.01", "device.n.01", "room.n.01"], "f_hard": ["person.n.01", "animal.n.01", "sport.n.01"], "f_hard_n": 2, "r_easy": ["husband", "matador", "bulldog"], "r_hard": ["landlord", "ethnic", "deer"], "r_hard_n": 2}, {"ent": "coachman", "cat": "worker.n.01", "cname": "worker", "f_easy": ["insect.n.01", "dish.n.02", "reptile.n.01"], "f_hard": ["athlete.n.01", "artist.n.01", "insect.n.01"], "f_hard_n": 2, "r_easy": ["undercarriage", "greenhouse", "fife"], "r_hard": ["cricketer", "modernist", "paranoid"], "r_hard_n": 2}, {"ent": "coauthor", "cat": "person.n.01", "cname": "person", "f_easy": ["sport.n.01", "game.n.01", "room.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "aircraft.n.01"], "f_hard_n": 2, "r_easy": ["chisel", "syringe", "camellia"], "r_hard": ["succulent", "bot", "undies"], "r_hard_n": 2}, {"ent": "coffin", "cat": "container.n.01", "cname": "container", "f_easy": ["room.n.01", "fabric.n.01", "tool.n.01"], "f_hard": ["fabric.n.01", "tool.n.01", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["equatorial", "mongoose", "chisel"], "r_hard": ["forceps", "caliper", "recliner"], "r_hard_n": 3}, {"ent": "collie", "cat": "dog.n.01", "cname": "dog", "f_easy": ["person.n.01", "tool.n.01", "musical_instrument.n.01"], "f_hard": ["dish.n.02", "metallic_element.n.01", "device.n.01"], "f_hard_n": 0, "r_easy": ["dauphin", "brandy", "immigrant"], "r_hard": ["piccolo", "philanthropist", "amplifier"], "r_hard_n": 0}, {"ent": "commissar", "cat": "worker.n.01", "cname": "worker", "f_easy": ["game.n.01", "animal.n.01", "disease.n.01"], "f_hard": ["artist.n.01", "athlete.n.01", "flower.n.01"], "f_hard_n": 2, "r_easy": ["stockman", "veronica", "falafel"], "r_hard": ["illustrator", "modernist", "grandparent"], "r_hard_n": 2}, {"ent": "composer", "cat": "artist.n.01", "cname": "artist", "f_easy": ["weapon.n.01", "clothing.n.01", "food.n.01"], "f_hard": ["worker.n.01", "athlete.n.01", "dish.n.02"], "f_hard_n": 2, "r_easy": ["probable", "matador", "briefcase"], "r_hard": ["salesperson", "boatman", "stingray"], "r_hard_n": 2}, {"ent": "conch", "cat": "animal.n.01", "cname": "animal", "f_easy": ["equipment.n.01", "musical_instrument.n.01", "artifact.n.01"], "f_hard": ["person.n.01", "plant.n.02", "artifact.n.01"], "f_hard_n": 2, "r_easy": ["rustic", "attacker", "undercarriage"], "r_hard": ["moll", "grader", "paintbrush"], "r_hard_n": 2}, {"ent": "condor", "cat": "bird.n.01", "cname": "bird", "f_easy": ["artifact.n.01", "food.n.01", "dog.n.01"], "f_hard": ["fish.n.01", "reptile.n.01", "mammal.n.01"], "f_hard_n": 3, "r_easy": ["larva", "janitor", "plow"], "r_hard": ["rhinoceros", "tortoise", "baboon"], "r_hard_n": 3}, {"ent": "confidante", "cat": "person.n.01", "cname": "person", "f_easy": ["dog.n.01", "insect.n.01", "musical_instrument.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "equipment.n.01"], "f_hard_n": 2, "r_easy": ["chisel", "stork", "succulent"], "r_hard": ["senna", "bot", "forceps"], "r_hard_n": 2}, {"ent": "conger", "cat": "fish.n.01", "cname": "fish", "f_easy": ["vehicle.n.01", "dog.n.01", "dish.n.02"], "f_hard": ["insect.n.01", "bird.n.01", "mammal.n.01"], "f_hard_n": 3, "r_easy": ["hoarder", "conch", "beagle"], "r_hard": ["mustang", "finch", "deer"], "r_hard_n": 3}, {"ent": "contraceptive", "cat": "device.n.01", "cname": "device", "f_easy": ["person.n.01", "food.n.01", "material.n.01"], "f_hard": ["equipment.n.01", "container.n.01", "furniture.n.01"], "f_hard_n": 3, "r_easy": ["cutthroat", "migrant", "sectional"], "r_hard": ["pagoda", "coffin", "minster"], "r_hard_n": 3}, {"ent": "convalescent", "cat": "person.n.01", "cname": "person", "f_easy": ["artifact.n.01", "organ.n.01", "boat.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "artifact.n.01"], "f_hard_n": 2, "r_easy": ["camellia", "undercarriage", "deflector"], "r_hard": ["azalea", "wildflower", "poncho"], "r_hard_n": 2}, {"ent": "convert", "cat": "person.n.01", "cname": "person", "f_easy": ["boat.n.01", "animal.n.01", "fabric.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "building.n.01"], "f_hard_n": 2, "r_easy": ["hyena", "mastiff", "loom"], "r_hard": ["larva", "foal", "potion"], "r_hard_n": 2}, {"ent": "corgi", "cat": "dog.n.01", "cname": "dog", "f_easy": ["boat.n.01", "artist.n.01", "building.n.01"], "f_hard": ["device.n.01", "athlete.n.01", "car.n.01"], "f_hard_n": 0, "r_easy": ["gavel", "sarin", "peripheral"], "r_hard": ["welder", "ridley", "underdog"], "r_hard_n": 0}, {"ent": "cougar", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["room.n.01", "car.n.01", "tool.n.01"], "f_hard": ["insect.n.01", "fish.n.01", "bird.n.01"], "f_hard_n": 3, "r_easy": ["teapot", "stepfather", "exhibitor"], "r_hard": ["chub", "finch", "bantam"], "r_hard_n": 3}, {"ent": "councilman", "cat": "person.n.01", "cname": "person", "f_easy": ["device.n.01", "building.n.01", "boat.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "edible_fruit.n.01"], "f_hard_n": 2, "r_easy": ["tun", "mangrove", "ridley"], "r_hard": ["clary", "clover", "bayonet"], "r_hard_n": 2}, {"ent": "countess", "cat": "person.n.01", "cname": "person", "f_easy": ["vehicle.n.01", "dish.n.02", "car.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "machine.n.01"], "f_hard_n": 2, "r_easy": ["moped", "candlestick", "integrator"], "r_hard": ["yucca", "senna", "skiff"], "r_hard_n": 2}, {"ent": "coupe", "cat": "car.n.01", "cname": "car", "f_easy": ["insect.n.01", "metallic_element.n.01", "aircraft.n.01"], "f_hard": ["metallic_element.n.01", "equipment.n.01", "fish.n.01"], "f_hard_n": 0, "r_easy": ["breeder", "vise", "incurable"], "r_hard": ["briefcase", "vise", "warden"], "r_hard_n": 0}, {"ent": "cowgirl", "cat": "worker.n.01", "cname": "worker", "f_easy": ["artist.n.01", "plant.n.02", "material.n.01"], "f_hard": ["artist.n.01", "athlete.n.01", "footwear.n.02"], "f_hard_n": 2, "r_easy": ["incurable", "godmother", "dropper"], "r_hard": ["composer", "illustrator", "paisley"], "r_hard_n": 2}, {"ent": "cricketer", "cat": "athlete.n.01", "cname": "athlete", "f_easy": ["machine.n.01", "bird.n.01", "fabric.n.01"], "f_hard": ["artist.n.01", "worker.n.01", "tool.n.01"], "f_hard_n": 2, "r_easy": ["geisha", "wristwatch", "paintbrush"], "r_hard": ["housekeeper", "waitress", "juggler"], "r_hard_n": 2}, {"ent": "crocodile", "cat": "reptile.n.01", "cname": "reptile", "f_easy": ["plaything.n.01", "flower.n.01", "fish.n.01"], "f_hard": ["bird.n.01", "fish.n.01", "mammal.n.01"], "f_hard_n": 3, "r_easy": ["kicker", "gland", "telegraph"], "r_hard": ["mako", "chimpanzee", "finch"], "r_hard_n": 3}, {"ent": "crossbow", "cat": "weapon.n.01", "cname": "weapon", "f_easy": ["artist.n.01", "plaything.n.01", "machine.n.01"], "f_hard": ["musical_instrument.n.01", "machine.n.01", "tool.n.01"], "f_hard_n": 2, "r_easy": ["bulldozer", "statesman", "haggis"], "r_hard": ["cymbal", "dynamo", "valet"], "r_hard_n": 2}, {"ent": "curry", "cat": "dish.n.02", "cname": "dish", "f_easy": ["plaything.n.01", "vegetable.n.01", "sport.n.01"], "f_hard": ["beverage.n.01", "dog.n.01", "building.n.01"], "f_hard_n": 1, "r_easy": ["treasurer", "bumblebee", "peregrine"], "r_hard": ["margarita", "dynamo", "mustang"], "r_hard_n": 1}, {"ent": "customer", "cat": "person.n.01", "cname": "person", "f_easy": ["car.n.01", "metallic_element.n.01", "organ.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "insect.n.01"], "f_hard_n": 2, "r_easy": ["falafel", "beagle", "camellia"], "r_hard": ["bramble", "mesquite", "turbine"], "r_hard_n": 2}, {"ent": "cutthroat", "cat": "person.n.01", "cname": "person", "f_easy": ["building.n.01", "fabric.n.01", "container.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "fish.n.01"], "f_hard_n": 2, "r_easy": ["telegraph", "burrito", "dimmer"], "r_hard": ["mariposa", "wildflower", "koala"], "r_hard_n": 2}, {"ent": "cymbal", "cat": "musical_instrument.n.01", "cname": "musical instrument", "f_easy": ["mammal.n.01", "insect.n.01", "dog.n.01"], "f_hard": ["weapon.n.01", "machine.n.01", "beverage.n.01"], "f_hard_n": 2, "r_easy": ["clarence", "dongle", "vaporizer"], "r_hard": ["gat", "rapier", "speakeasy"], "r_hard_n": 2}, {"ent": "dachshund", "cat": "dog.n.01", "cname": "dog", "f_easy": ["game.n.01", "fish.n.01", "person.n.01"], "f_hard": ["clothing.n.01", "building.n.01", "fabric.n.01"], "f_hard_n": 0, "r_easy": ["backpack", "sarcophagus", "accelerometer"], "r_hard": ["curry", "yacht", "sherry"], "r_hard_n": 0}, {"ent": "darkroom", "cat": "room.n.01", "cname": "room", "f_easy": ["material.n.01", "worker.n.01", "beverage.n.01"], "f_hard": ["container.n.01", "fabric.n.01", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["peregrine", "warrior", "educator"], "r_hard": ["caliper", "waveguide", "sofa"], "r_hard_n": 3}, {"ent": "dauphin", "cat": "person.n.01", "cname": "person", "f_easy": ["edible_fruit.n.01", "car.n.01", "building.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "container.n.01"], "f_hard_n": 2, "r_easy": ["cymbal", "necktie", "dug"], "r_hard": ["clary", "stray", "equatorial"], "r_hard_n": 2}, {"ent": "debtor", "cat": "person.n.01", "cname": "person", "f_easy": ["material.n.01", "organ.n.01", "fish.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "organ.n.01"], "f_hard_n": 2, "r_easy": ["ax", "betel", "starship"], "r_hard": ["mammal", "buttercup", "limousine"], "r_hard_n": 2}, {"ent": "deer", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["organ.n.01", "plant.n.02", "weapon.n.01"], "f_hard": ["reptile.n.01", "insect.n.01", "fish.n.01"], "f_hard_n": 3, "r_easy": ["thwart", "coupe", "wraparound"], "r_hard": ["woodpecker", "tortoise", "pelican"], "r_hard_n": 3}, {"ent": "deflector", "cat": "device.n.01", "cname": "device", "f_easy": ["metallic_element.n.01", "artist.n.01", "organism.n.01"], "f_hard": ["container.n.01", "room.n.01", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["spastic", "preteen", "landlord"], "r_hard": ["backpack", "toby", "telegraph"], "r_hard_n": 3}, {"ent": "delinquent", "cat": "person.n.01", "cname": "person", "f_easy": ["dish.n.02", "animal.n.01", "dog.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "fabric.n.01"], "f_hard_n": 2, "r_easy": ["valerian", "stray", "buttercup"], "r_hard": ["larva", "fuchsia", "cesspool"], "r_hard_n": 2}, {"ent": "denture", "cat": "device.n.01", "cname": "device", "f_easy": ["body_part.n.01", "vegetable.n.01", "vehicle.n.01"], "f_hard": ["building.n.01", "container.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["ballerina", "minnow", "moray"], "r_hard": ["towel", "sectional", "yacht"], "r_hard_n": 3}, {"ent": "dermatologist", "cat": "person.n.01", "cname": "person", "f_easy": ["reptile.n.01", "material.n.01", "game.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "vegetable.n.01"], "f_hard_n": 2, "r_easy": ["mamba", "soccer", "brad"], "r_hard": ["althea", "jasmine", "clarence"], "r_hard_n": 2}, {"ent": "desk", "cat": "furniture.n.01", "cname": "furniture", "f_easy": ["tool.n.01", "organ.n.01", "athlete.n.01"], "f_hard": ["fabric.n.01", "vehicle.n.01", "building.n.01"], "f_hard_n": 3, "r_easy": ["supremacist", "spandex", "toddler"], "r_hard": ["bulldozer", "yacht", "maul"], "r_hard_n": 3}, {"ent": "dimmer", "cat": "device.n.01", "cname": "device", "f_easy": ["room.n.01", "boat.n.01", "plant.n.02"], "f_hard": ["footwear.n.02", "plaything.n.01", "tool.n.01"], "f_hard_n": 3, "r_easy": ["housemate", "cardiologist", "matador"], "r_hard": ["panzer", "wraparound", "sheeting"], "r_hard_n": 3}, {"ent": "dinky", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["dish.n.02", "animal.n.01", "athlete.n.01"], "f_hard": ["equipment.n.01", "tool.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["pickpocket", "scion", "crocodile"], "r_hard": ["integrator", "desk", "ax"], "r_hard_n": 3}, {"ent": "dinky", "cat": "container.n.01", "cname": "container", "f_easy": ["edible_fruit.n.01", "mammal.n.01", "metallic_element.n.01"], "f_hard": ["furniture.n.01", "fabric.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["investor", "untouchable", "sprinkler"], "r_hard": ["transponder", "dreadnought", "peripheral"], "r_hard_n": 3}, {"ent": "dirk", "cat": "weapon.n.01", "cname": "weapon", "f_easy": ["athlete.n.01", "flower.n.01", "furniture.n.01"], "f_hard": ["musical_instrument.n.01", "machine.n.01", "bird.n.01"], "f_hard_n": 2, "r_easy": ["whiskey", "calender", "liza"], "r_hard": ["cymbal", "harmonica", "therapist"], "r_hard_n": 2}, {"ent": "dongle", "cat": "device.n.01", "cname": "device", "f_easy": ["equipment.n.01", "athlete.n.01", "food.n.01"], "f_hard": ["plaything.n.01", "clothing.n.01", "building.n.01"], "f_hard_n": 3, "r_easy": ["avenger", "stingray", "waitress"], "r_hard": ["scythe", "georgette", "poncho"], "r_hard_n": 3}, {"ent": "dowager", "cat": "person.n.01", "cname": "person", "f_easy": ["beverage.n.01", "dish.n.02", "machine.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "fish.n.01"], "f_hard_n": 2, "r_easy": ["capacitor", "accelerometer", "equatorial"], "r_hard": ["biennial", "stray", "cesspool"], "r_hard_n": 2}, {"ent": "dragoon", "cat": "worker.n.01", "cname": "worker", "f_easy": ["plaything.n.01", "sport.n.01", "fish.n.01"], "f_hard": ["athlete.n.01", "artist.n.01", "body_part.n.01"], "f_hard_n": 2, "r_easy": ["caribou", "classroom", "invalid"], "r_hard": ["modernist", "surrealist", "rowan"], "r_hard_n": 2}, {"ent": "dreadnought", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["vegetable.n.01", "reptile.n.01", "metallic_element.n.01"], "f_hard": ["device.n.01", "furniture.n.01", "clothing.n.01"], "f_hard_n": 3, "r_easy": ["puppeteer", "matador", "tartan"], "r_hard": ["scraper", "beanie", "poncho"], "r_hard_n": 3}, {"ent": "dredge", "cat": "machine.n.01", "cname": "machine", "f_easy": ["dish.n.02", "tool.n.01", "mammal.n.01"], "f_hard": ["weapon.n.01", "musical_instrument.n.01", "reptile.n.01"], "f_hard_n": 2, "r_easy": ["semaphore", "anesthesiologist", "transponder"], "r_hard": ["dirk", "fife", "minnow"], "r_hard_n": 2}, {"ent": "dropper", "cat": "device.n.01", "cname": "device", "f_easy": ["reptile.n.01", "material.n.01", "room.n.01"], "f_hard": ["container.n.01", "tool.n.01", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["amplifier", "adjuster", "customer"], "r_hard": ["peripheral", "spandex", "ship"], "r_hard_n": 3}, {"ent": "dubbing", "cat": "device.n.01", "cname": "device", "f_easy": ["food.n.01", "material.n.01", "plaything.n.01"], "f_hard": ["building.n.01", "clothing.n.01", "fabric.n.01"], "f_hard_n": 3, "r_easy": ["urchin", "bulldozer", "mustang"], "r_hard": ["backpack", "classroom", "scraper"], "r_hard_n": 3}, {"ent": "dug", "cat": "organ.n.01", "cname": "organ", "f_easy": ["disease.n.01", "plaything.n.01", "vegetable.n.01"], "f_hard": ["flower.n.01", "mammal.n.01", "vehicle.n.01"], "f_hard_n": 0, "r_easy": ["accountant", "chronicler", "bumblebee"], "r_hard": ["liza", "swine", "tweeter"], "r_hard_n": 0}, {"ent": "dynamo", "cat": "machine.n.01", "cname": "machine", "f_easy": ["building.n.01", "edible_fruit.n.01", "aircraft.n.01"], "f_hard": ["musical_instrument.n.01", "weapon.n.01", "tool.n.01"], "f_hard_n": 2, "r_easy": ["flashlight", "dreadnought", "ballerina"], "r_hard": ["shiv", "harmonica", "equatorial"], "r_hard_n": 2}, {"ent": "eclectic", "cat": "person.n.01", "cname": "person", "f_easy": ["weapon.n.01", "vegetable.n.01", "device.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "fish.n.01"], "f_hard_n": 2, "r_easy": ["mammal", "stapler", "pinto"], "r_hard": ["wildflower", "foal", "dredge"], "r_hard_n": 2}, {"ent": "educator", "cat": "person.n.01", "cname": "person", "f_easy": ["plaything.n.01", "organ.n.01", "vegetable.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "tool.n.01"], "f_hard_n": 2, "r_easy": ["burrito", "accelerometer", "sherry"], "r_hard": ["mariposa", "foal", "bistro"], "r_hard_n": 2}, {"ent": "egalitarian", "cat": "person.n.01", "cname": "person", "f_easy": ["disease.n.01", "game.n.01", "metallic_element.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "device.n.01"], "f_hard_n": 2, "r_easy": ["lilac", "desk", "speakeasy"], "r_hard": ["mammal", "conch", "stapler"], "r_hard_n": 2}, {"ent": "empress", "cat": "person.n.01", "cname": "person", "f_easy": ["vehicle.n.01", "dog.n.01", "room.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "body_part.n.01"], "f_hard_n": 2, "r_easy": ["bayonet", "tweeter", "dimmer"], "r_hard": ["wildflower", "succulent", "wraparound"], "r_hard_n": 2}, {"ent": "environmentalist", "cat": "person.n.01", "cname": "person", "f_easy": ["tool.n.01", "game.n.01", "body_part.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "bird.n.01"], "f_hard_n": 2, "r_easy": ["dug", "wildflower", "crocodile"], "r_hard": ["bot", "senna", "swine"], "r_hard_n": 2}, {"ent": "equatorial", "cat": "device.n.01", "cname": "device", "f_easy": ["sport.n.01", "tool.n.01", "bird.n.01"], "f_hard": ["footwear.n.02", "building.n.01", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["capo", "fascist", "marc"], "r_hard": ["boathouse", "suiting", "tavern"], "r_hard_n": 3}, {"ent": "erica", "cat": "plant.n.02", "cname": "plant", "f_easy": ["aircraft.n.01", "clothing.n.01", "tool.n.01"], "f_hard": ["animal.n.01", "person.n.01", "weapon.n.01"], "f_hard_n": 2, "r_easy": ["sofa", "weatherman", "betrothed"], "r_hard": ["celibate", "bimbo", "scion"], "r_hard_n": 2}, {"ent": "espresso", "cat": "beverage.n.01", "cname": "drink", "f_easy": ["aircraft.n.01", "game.n.01", "disease.n.01"], "f_hard": ["dish.n.02", "edible_fruit.n.01", "container.n.01"], "f_hard_n": 1, "r_easy": ["shaheed", "scythe", "ancestor"], "r_hard": ["falafel", "redneck", "misfit"], "r_hard_n": 1}, {"ent": "ethnic", "cat": "person.n.01", "cname": "person", "f_easy": ["insect.n.01", "organ.n.01", "machine.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "reptile.n.01"], "f_hard_n": 2, "r_easy": ["incubator", "belting", "cougar"], "r_hard": ["bramble", "newt", "collie"], "r_hard_n": 2}, {"ent": "evergreen", "cat": "plant.n.02", "cname": "plant", "f_easy": ["fish.n.01", "metallic_element.n.01", "material.n.01"], "f_hard": ["person.n.01", "animal.n.01", "weapon.n.01"], "f_hard_n": 2, "r_easy": ["thwart", "cowgirl", "heathen"], "r_hard": ["survivalist", "misfit", "amplifier"], "r_hard_n": 2}, {"ent": "exhibitor", "cat": "person.n.01", "cname": "person", "f_easy": ["dish.n.02", "edible_fruit.n.01", "tree.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "organ.n.01"], "f_hard_n": 2, "r_easy": ["potion", "pontifical", "shim"], "r_hard": ["jasmine", "wildflower", "cymbal"], "r_hard_n": 2}, {"ent": "exporter", "cat": "person.n.01", "cname": "person", "f_easy": ["furniture.n.01", "clothing.n.01", "musical_instrument.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "building.n.01"], "f_hard_n": 2, "r_easy": ["shed", "wombat", "hooch"], "r_hard": ["newt", "fuchsia", "zebra"], "r_hard_n": 2}, {"ent": "extrovert", "cat": "person.n.01", "cname": "person", "f_easy": ["game.n.01", "dish.n.02", "fish.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "artifact.n.01"], "f_hard_n": 2, "r_easy": ["shiv", "tabor", "crossbow"], "r_hard": ["starfish", "conch", "bedroom"], "r_hard_n": 2}, {"ent": "falafel", "cat": "dish.n.02", "cname": "dish", "f_easy": ["device.n.01", "dog.n.01", "aircraft.n.01"], "f_hard": ["beverage.n.01", "tree.n.01", "container.n.01"], "f_hard_n": 1, "r_easy": ["georgette", "dachshund", "brandy"], "r_hard": ["sherry", "countess", "calender"], "r_hard_n": 1}, {"ent": "falcon", "cat": "bird.n.01", "cname": "bird", "f_easy": ["body_part.n.01", "reptile.n.01", "disease.n.01"], "f_hard": ["mammal.n.01", "insect.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["exhibitor", "informer", "isolationist"], "r_hard": ["chipmunk", "deer", "stingray"], "r_hard_n": 3}, {"ent": "fancier", "cat": "person.n.01", "cname": "person", "f_easy": ["clothing.n.01", "tree.n.01", "insect.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "footwear.n.02"], "f_hard_n": 2, "r_easy": ["coffin", "tongs", "chimpanzee"], "r_hard": ["succulent", "biennial", "hummingbird"], "r_hard_n": 2}, {"ent": "fascist", "cat": "person.n.01", "cname": "person", "f_easy": ["dish.n.02", "footwear.n.02", "musical_instrument.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "artifact.n.01"], "f_hard_n": 2, "r_easy": ["recliner", "houseboat", "moped"], "r_hard": ["jasmine", "piglet", "succulent"], "r_hard_n": 2}, {"ent": "fife", "cat": "musical_instrument.n.01", "cname": "musical instrument", "f_easy": ["flower.n.01", "material.n.01", "room.n.01"], "f_hard": ["machine.n.01", "weapon.n.01", "aircraft.n.01"], "f_hard_n": 2, "r_easy": ["classroom", "waveguide", "archbishop"], "r_hard": ["dirk", "turbine", "chub"], "r_hard_n": 2}, {"ent": "finch", "cat": "bird.n.01", "cname": "bird", "f_easy": ["game.n.01", "reptile.n.01", "tree.n.01"], "f_hard": ["insect.n.01", "fish.n.01", "mammal.n.01"], "f_hard_n": 3, "r_easy": ["dragoon", "capo", "clinician"], "r_hard": ["chimpanzee", "ridley", "wombat"], "r_hard_n": 3}, {"ent": "flamingo", "cat": "bird.n.01", "cname": "bird", "f_easy": ["body_part.n.01", "sport.n.01", "machine.n.01"], "f_hard": ["mammal.n.01", "reptile.n.01", "insect.n.01"], "f_hard_n": 3, "r_easy": ["senator", "workstation", "heathen"], "r_hard": ["mustang", "pinto", "liza"], "r_hard_n": 3}, {"ent": "flashlight", "cat": "device.n.01", "cname": "device", "f_easy": ["equipment.n.01", "organ.n.01", "room.n.01"], "f_hard": ["equipment.n.01", "room.n.01", "clothing.n.01"], "f_hard_n": 3, "r_easy": ["grocer", "minivan", "wheelchair"], "r_hard": ["dinky", "recliner", "sickle"], "r_hard_n": 3}, {"ent": "foal", "cat": "animal.n.01", "cname": "animal", "f_easy": ["building.n.01", "disease.n.01", "clothing.n.01"], "f_hard": ["person.n.01", "plant.n.02", "athlete.n.01"], "f_hard_n": 2, "r_easy": ["pasha", "brandy", "pontifical"], "r_hard": ["rustic", "capo", "ship"], "r_hard_n": 2}, {"ent": "forceps", "cat": "device.n.01", "cname": "device", "f_easy": ["dish.n.02", "food.n.01", "organism.n.01"], "f_hard": ["container.n.01", "building.n.01", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["burrito", "genitalia", "psychiatrist"], "r_hard": ["sofa", "sheeting", "bicycle"], "r_hard_n": 3}, {"ent": "fuchsia", "cat": "plant.n.02", "cname": "plant", "f_easy": ["fish.n.01", "clothing.n.01", "aircraft.n.01"], "f_hard": ["animal.n.01", "person.n.01", "equipment.n.01"], "f_hard_n": 2, "r_easy": ["wheelchair", "voyeur", "gasket"], "r_hard": ["hillbilly", "narc", "sarcophagus"], "r_hard_n": 2}, {"ent": "gasket", "cat": "device.n.01", "cname": "device", "f_easy": ["athlete.n.01", "organism.n.01", "aircraft.n.01"], "f_hard": ["clothing.n.01", "fabric.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["protectionist", "betel", "dauphin"], "r_hard": ["classroom", "manger", "churn"], "r_hard_n": 3}, {"ent": "gat", "cat": "weapon.n.01", "cname": "weapon", "f_easy": ["flower.n.01", "person.n.01", "food.n.01"], "f_hard": ["musical_instrument.n.01", "machine.n.01", "room.n.01"], "f_hard_n": 2, "r_easy": ["helicopter", "nawab", "portable"], "r_hard": ["stapler", "piccolo", "turbine"], "r_hard_n": 2}, {"ent": "gavel", "cat": "tool.n.01", "cname": "tool", "f_easy": ["metallic_element.n.01", "edible_fruit.n.01", "dish.n.02"], "f_hard": ["equipment.n.01", "fabric.n.01", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["steeper", "gasket", "toddler"], "r_hard": ["undies", "shirt", "syringe"], "r_hard_n": 3}, {"ent": "geisha", "cat": "person.n.01", "cname": "person", "f_easy": ["flower.n.01", "device.n.01", "equipment.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "fish.n.01"], "f_hard_n": 2, "r_easy": ["meatball", "wombat", "bot"], "r_hard": ["althea", "succulent", "meatball"], "r_hard_n": 2}, {"ent": "genitalia", "cat": "organ.n.01", "cname": "organ", "f_easy": ["metallic_element.n.01", "room.n.01", "athlete.n.01"], "f_hard": ["mammal.n.01", "artist.n.01", "athlete.n.01"], "f_hard_n": 0, "r_easy": ["yucca", "rodent", "liza"], "r_hard": ["importer", "homophobe", "starfish"], "r_hard_n": 0}, {"ent": "georgette", "cat": "fabric.n.01", "cname": "fabric", "f_easy": ["car.n.01", "artist.n.01", "mammal.n.01"], "f_hard": ["footwear.n.02", "equipment.n.01", "building.n.01"], "f_hard_n": 3, "r_easy": ["janitor", "mangrove", "snoop"], "r_hard": ["wristwatch", "steeper", "minster"], "r_hard_n": 3}, {"ent": "girder", "cat": "device.n.01", "cname": "device", "f_easy": ["dog.n.01", "plaything.n.01", "vegetable.n.01"], "f_hard": ["fabric.n.01", "vehicle.n.01", "tool.n.01"], "f_hard_n": 3, "r_easy": ["avenger", "worthy", "poncho"], "r_hard": ["pagoda", "towel", "classroom"], "r_hard_n": 3}, {"ent": "gland", "cat": "organ.n.01", "cname": "organ", "f_easy": ["container.n.01", "worker.n.01", "footwear.n.02"], "f_hard": ["vehicle.n.01", "dish.n.02", "insect.n.01"], "f_hard_n": 0, "r_easy": ["waveguide", "convalescent", "narrator"], "r_hard": ["importer", "scraper", "handyman"], "r_hard_n": 0}, {"ent": "glider", "cat": "aircraft.n.01", "cname": "aircraft", "f_easy": ["metallic_element.n.01", "dog.n.01", "bird.n.01"], "f_hard": ["boat.n.01", "tree.n.01", "building.n.01"], "f_hard_n": 1, "r_easy": ["pug", "clockwork", "tavern"], "r_hard": ["skiff", "spastic", "panzer"], "r_hard_n": 1}, {"ent": "godmother", "cat": "person.n.01", "cname": "person", "f_easy": ["boat.n.01", "equipment.n.01", "room.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "material.n.01"], "f_hard_n": 2, "r_easy": ["paella", "portable", "dug"], "r_hard": ["evergreen", "wildflower", "waveguide"], "r_hard_n": 2}, {"ent": "goggles", "cat": "device.n.01", "cname": "device", "f_easy": ["organism.n.01", "material.n.01", "game.n.01"], "f_hard": ["building.n.01", "container.n.01", "clothing.n.01"], "f_hard_n": 3, "r_easy": ["librarian", "swami", "margarita"], "r_hard": ["beanie", "wraparound", "bistro"], "r_hard_n": 3}, {"ent": "goldfish", "cat": "fish.n.01", "cname": "fish", "f_easy": ["food.n.01", "edible_fruit.n.01", "tree.n.01"], "f_hard": ["insect.n.01", "reptile.n.01", "bird.n.01"], "f_hard_n": 3, "r_easy": ["cowgirl", "girder", "syringe"], "r_hard": ["wombat", "koala", "hummingbird"], "r_hard_n": 3}, {"ent": "grabber", "cat": "person.n.01", "cname": "person", "f_easy": ["vegetable.n.01", "device.n.01", "game.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "equipment.n.01"], "f_hard_n": 2, "r_easy": ["camellia", "erica", "portable"], "r_hard": ["larva", "senna", "brandy"], "r_hard_n": 2}, {"ent": "grader", "cat": "person.n.01", "cname": "person", "f_easy": ["disease.n.01", "organ.n.01", "insect.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "artifact.n.01"], "f_hard_n": 2, "r_easy": ["pagoda", "rowan", "poncho"], "r_hard": ["lilac", "mammal", "maggot"], "r_hard_n": 2}, {"ent": "granddaughter", "cat": "person.n.01", "cname": "person", "f_easy": ["sport.n.01", "fish.n.01", "equipment.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "weapon.n.01"], "f_hard_n": 2, "r_easy": ["piglet", "yacht", "paintbrush"], "r_hard": ["buttercup", "evergreen", "clarence"], "r_hard_n": 2}, {"ent": "grandma", "cat": "person.n.01", "cname": "person", "f_easy": ["edible_fruit.n.01", "plant.n.02", "weapon.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "equipment.n.01"], "f_hard_n": 2, "r_easy": ["althea", "woodpecker", "cello"], "r_hard": ["stray", "buttercup", "banjo"], "r_hard_n": 2}, {"ent": "grandparent", "cat": "person.n.01", "cname": "person", "f_easy": ["dog.n.01", "musical_instrument.n.01", "fish.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "dog.n.01"], "f_hard_n": 2, "r_easy": ["foal", "tongs", "brogan"], "r_hard": ["wildflower", "piglet", "mariposa"], "r_hard_n": 2}, {"ent": "grandson", "cat": "person.n.01", "cname": "person", "f_easy": ["disease.n.01", "aircraft.n.01", "food.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "artifact.n.01"], "f_hard_n": 2, "r_easy": ["ship", "peregrine", "wristwatch"], "r_hard": ["mammal", "starfish", "krill"], "r_hard_n": 2}, {"ent": "greenhouse", "cat": "building.n.01", "cname": "building", "f_easy": ["dog.n.01", "equipment.n.01", "game.n.01"], "f_hard": ["fabric.n.01", "plaything.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["rayon", "retailer", "underdog"], "r_hard": ["amplifier", "gavel", "integrator"], "r_hard_n": 3}, {"ent": "greyhound", "cat": "dog.n.01", "cname": "dog", "f_easy": ["plant.n.02", "edible_fruit.n.01", "device.n.01"], "f_hard": ["sport.n.01", "tree.n.01", "vehicle.n.01"], "f_hard_n": 0, "r_easy": ["pulley", "absentee", "grandparent"], "r_hard": ["radiologist", "mahatma", "anesthesiologist"], "r_hard_n": 0}, {"ent": "grocer", "cat": "person.n.01", "cname": "person", "f_easy": ["machine.n.01", "fish.n.01", "tool.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "container.n.01"], "f_hard_n": 2, "r_easy": ["krill", "deer", "clover"], "r_hard": ["succulent", "mesquite", "newt"], "r_hard_n": 2}, {"ent": "haggis", "cat": "dish.n.02", "cname": "dish", "f_easy": ["clothing.n.01", "artifact.n.01", "reptile.n.01"], "f_hard": ["beverage.n.01", "game.n.01", "clothing.n.01"], "f_hard_n": 1, "r_easy": ["limousine", "juggler", "bodybuilder"], "r_hard": ["potion", "aquarium", "neutral"], "r_hard_n": 1}, {"ent": "handcuff", "cat": "device.n.01", "cname": "device", "f_easy": ["aircraft.n.01", "tool.n.01", "vehicle.n.01"], "f_hard": ["fabric.n.01", "room.n.01", "furniture.n.01"], "f_hard_n": 3, "r_easy": ["undergraduate", "customer", "quitter"], "r_hard": ["briefcase", "semaphore", "undies"], "r_hard_n": 3}, {"ent": "handyman", "cat": "worker.n.01", "cname": "worker", "f_easy": ["animal.n.01", "building.n.01", "clothing.n.01"], "f_hard": ["athlete.n.01", "artist.n.01", "plaything.n.01"], "f_hard_n": 2, "r_easy": ["sieve", "totalitarian", "liberator"], "r_hard": ["modernist", "kicker", "egalitarian"], "r_hard_n": 2}, {"ent": "hangman", "cat": "person.n.01", "cname": "person", "f_easy": ["building.n.01", "footwear.n.02", "dog.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "plaything.n.01"], "f_hard_n": 2, "r_easy": ["coffin", "veronica", "loom"], "r_hard": ["althea", "piglet", "sickle"], "r_hard_n": 2}, {"ent": "harmonica", "cat": "musical_instrument.n.01", "cname": "musical instrument", "f_easy": ["machine.n.01", "metallic_element.n.01", "plaything.n.01"], "f_hard": ["weapon.n.01", "machine.n.01", "reptile.n.01"], "f_hard_n": 2, "r_easy": ["delinquent", "psychiatrist", "adjuster"], "r_hard": ["gat", "turbine", "bedroom"], "r_hard_n": 2}, {"ent": "harrow", "cat": "machine.n.01", "cname": "machine", "f_easy": ["car.n.01", "person.n.01", "clothing.n.01"], "f_hard": ["musical_instrument.n.01", "weapon.n.01", "artist.n.01"], "f_hard_n": 2, "r_easy": ["environmentalist", "meatball", "supermodel"], "r_hard": ["rifle", "sarin", "sabra"], "r_hard_n": 2}, {"ent": "headmaster", "cat": "person.n.01", "cname": "person", "f_easy": ["aircraft.n.01", "fabric.n.01", "animal.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "clothing.n.01"], "f_hard_n": 2, "r_easy": ["cougar", "houseboat", "mangrove"], "r_hard": ["jasmine", "starfish", "conger"], "r_hard_n": 2}, {"ent": "heathen", "cat": "person.n.01", "cname": "person", "f_easy": ["container.n.01", "plant.n.02", "material.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "insect.n.01"], "f_hard_n": 2, "r_easy": ["handcuff", "dredge", "desk"], "r_hard": ["larva", "clover", "towel"], "r_hard_n": 2}, {"ent": "helicopter", "cat": "aircraft.n.01", "cname": "aircraft", "f_easy": ["insect.n.01", "tool.n.01", "building.n.01"], "f_hard": ["boat.n.01", "metallic_element.n.01", "weapon.n.01"], "f_hard_n": 1, "r_easy": ["rogue", "missus", "stepfather"], "r_hard": ["skiff", "starship", "gavel"], "r_hard_n": 1}, {"ent": "hillbilly", "cat": "person.n.01", "cname": "person", "f_easy": ["tree.n.01", "plaything.n.01", "car.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "organ.n.01"], "f_hard_n": 2, "r_easy": ["moped", "cheetah", "paintbrush"], "r_hard": ["clary", "jasmine", "dreadnought"], "r_hard_n": 2}, {"ent": "hoarder", "cat": "person.n.01", "cname": "person", "f_easy": ["food.n.01", "metallic_element.n.01", "fish.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "material.n.01"], "f_hard_n": 2, "r_easy": ["mastiff", "ax", "rifle"], "r_hard": ["clover", "wildflower", "stapler"], "r_hard_n": 2}, {"ent": "homophobe", "cat": "person.n.01", "cname": "person", "f_easy": ["musical_instrument.n.01", "tool.n.01", "edible_fruit.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "machine.n.01"], "f_hard_n": 2, "r_easy": ["incubator", "mariposa", "chisel"], "r_hard": ["azalea", "maggot", "helicopter"], "r_hard_n": 2}, {"ent": "hooch", "cat": "beverage.n.01", "cname": "drink", "f_easy": ["person.n.01", "dog.n.01", "dish.n.02"], "f_hard": ["dish.n.02", "edible_fruit.n.01", "artifact.n.01"], "f_hard_n": 1, "r_easy": ["shim", "exporter", "shaheed"], "r_hard": ["poi", "clarence", "injector"], "r_hard_n": 1}, {"ent": "houseboat", "cat": "boat.n.01", "cname": "boat", "f_easy": ["organism.n.01", "building.n.01", "footwear.n.02"], "f_hard": ["aircraft.n.01", "tool.n.01", "clothing.n.01"], "f_hard_n": 1, "r_easy": ["backpack", "erica", "grandma"], "r_hard": ["glider", "gasket", "coffin"], "r_hard_n": 1}, {"ent": "housekeeper", "cat": "worker.n.01", "cname": "worker", "f_easy": ["musical_instrument.n.01", "equipment.n.01", "organ.n.01"], "f_hard": ["athlete.n.01", "artist.n.01", "plaything.n.01"], "f_hard_n": 2, "r_easy": ["wildflower", "brazier", "wanton"], "r_hard": ["composer", "surrealist", "takin"], "r_hard_n": 2}, {"ent": "housemate", "cat": "person.n.01", "cname": "person", "f_easy": ["furniture.n.01", "footwear.n.02", "body_part.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "body_part.n.01"], "f_hard_n": 2, "r_easy": ["evergreen", "larva", "gavel"], "r_hard": ["buttercup", "stray", "biennial"], "r_hard_n": 2}, {"ent": "huck", "cat": "fabric.n.01", "cname": "fabric", "f_easy": ["disease.n.01", "tool.n.01", "flower.n.01"], "f_hard": ["equipment.n.01", "device.n.01", "building.n.01"], "f_hard_n": 3, "r_easy": ["aristocrat", "mahatma", "betrothed"], "r_hard": ["moped", "shim", "clarence"], "r_hard_n": 3}, {"ent": "hummingbird", "cat": "bird.n.01", "cname": "bird", "f_easy": ["artist.n.01", "metallic_element.n.01", "mammal.n.01"], "f_hard": ["mammal.n.01", "insect.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["yacht", "rhinoceros", "chateau"], "r_hard": ["koala", "llama", "hyena"], "r_hard_n": 3}, {"ent": "husband", "cat": "person.n.01", "cname": "person", "f_easy": ["fish.n.01", "machine.n.01", "artifact.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "footwear.n.02"], "f_hard_n": 2, "r_easy": ["scalpel", "swine", "dongle"], "r_hard": ["erica", "clover", "gavel"], "r_hard_n": 2}, {"ent": "hyena", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["plant.n.02", "furniture.n.01", "person.n.01"], "f_hard": ["insect.n.01", "fish.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["darkroom", "boathouse", "vise"], "r_hard": ["ridley", "bantam", "goldfish"], "r_hard_n": 3}, {"ent": "illustrator", "cat": "artist.n.01", "cname": "artist", "f_easy": ["organ.n.01", "aircraft.n.01", "flower.n.01"], "f_hard": ["worker.n.01", "athlete.n.01", "weapon.n.01"], "f_hard_n": 2, "r_easy": ["capo", "waveguide", "murine"], "r_hard": ["plumber", "dragoon", "aquarium"], "r_hard_n": 2}, {"ent": "immigrant", "cat": "person.n.01", "cname": "person", "f_easy": ["tree.n.01", "material.n.01", "edible_fruit.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "building.n.01"], "f_hard_n": 2, "r_easy": ["rhinoceros", "thwart", "stork"], "r_hard": ["starfish", "evergreen", "oboe"], "r_hard_n": 2}, {"ent": "immune", "cat": "person.n.01", "cname": "person", "f_easy": ["musical_instrument.n.01", "food.n.01", "machine.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "room.n.01"], "f_hard_n": 2, "r_easy": ["buggy", "sectional", "handcuff"], "r_hard": ["biennial", "maggot", "tugboat"], "r_hard_n": 2}, {"ent": "importer", "cat": "person.n.01", "cname": "person", "f_easy": ["vegetable.n.01", "tree.n.01", "device.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "plaything.n.01"], "f_hard_n": 2, "r_easy": ["cymbal", "coffin", "shiv"], "r_hard": ["clover", "evergreen", "crossbow"], "r_hard_n": 2}, {"ent": "incubator", "cat": "equipment.n.01", "cname": "equipment", "f_easy": ["room.n.01", "vehicle.n.01", "worker.n.01"], "f_hard": ["fabric.n.01", "vehicle.n.01", "building.n.01"], "f_hard_n": 3, "r_easy": ["greyhound", "tabor", "mako"], "r_hard": ["dinky", "tun", "brogan"], "r_hard_n": 3}, {"ent": "incurable", "cat": "person.n.01", "cname": "person", "f_easy": ["room.n.01", "bird.n.01", "vehicle.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "insect.n.01"], "f_hard_n": 2, "r_easy": ["murine", "sherry", "thwart"], "r_hard": ["biennial", "senna", "calender"], "r_hard_n": 2}, {"ent": "informer", "cat": "person.n.01", "cname": "person", "f_easy": ["organ.n.01", "flower.n.01", "tool.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "fish.n.01"], "f_hard_n": 2, "r_easy": ["rhinoceros", "semaphore", "minster"], "r_hard": ["mammal", "fuchsia", "storeroom"], "r_hard_n": 2}, {"ent": "injector", "cat": "device.n.01", "cname": "device", "f_easy": ["plant.n.02", "boat.n.01", "mammal.n.01"], "f_hard": ["footwear.n.02", "furniture.n.01", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["exhibitor", "meatball", "debtor"], "r_hard": ["georgette", "belting", "huck"], "r_hard_n": 3}, {"ent": "insider", "cat": "person.n.01", "cname": "person", "f_easy": ["device.n.01", "metallic_element.n.01", "musical_instrument.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "game.n.01"], "f_hard_n": 2, "r_easy": ["sarin", "jasmine", "soccer"], "r_hard": ["mariposa", "biennial", "tugboat"], "r_hard_n": 2}, {"ent": "integrator", "cat": "device.n.01", "cname": "device", "f_easy": ["game.n.01", "flower.n.01", "body_part.n.01"], "f_hard": ["room.n.01", "furniture.n.01", "fabric.n.01"], "f_hard_n": 3, "r_easy": ["secessionist", "totalitarian", "married"], "r_hard": ["aquarium", "cesspool", "suiting"], "r_hard_n": 3}, {"ent": "interceptor", "cat": "aircraft.n.01", "cname": "aircraft", "f_easy": ["footwear.n.02", "container.n.01", "plaything.n.01"], "f_hard": ["boat.n.01", "flower.n.01", "fish.n.01"], "f_hard_n": 1, "r_easy": ["grandson", "modernist", "countess"], "r_hard": ["tugboat", "therapist", "bobcat"], "r_hard_n": 1}, {"ent": "invalid", "cat": "person.n.01", "cname": "person", "f_easy": ["food.n.01", "machine.n.01", "tool.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "vegetable.n.01"], "f_hard_n": 2, "r_easy": ["woodpecker", "foal", "sandwich"], "r_hard": ["lilac", "clover", "loom"], "r_hard_n": 2}, {"ent": "investor", "cat": "person.n.01", "cname": "person", "f_easy": ["artifact.n.01", "insect.n.01", "material.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "bird.n.01"], "f_hard_n": 2, "r_easy": ["papillon", "genitalia", "falafel"], "r_hard": ["yucca", "newt", "recliner"], "r_hard_n": 2}, {"ent": "isolationist", "cat": "person.n.01", "cname": "person", "f_easy": ["building.n.01", "edible_fruit.n.01", "metallic_element.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "clothing.n.01"], "f_hard_n": 2, "r_easy": ["gasket", "crocodile", "bulldog"], "r_hard": ["bot", "piglet", "collie"], "r_hard_n": 2}, {"ent": "janitor", "cat": "person.n.01", "cname": "person", "f_easy": ["dog.n.01", "edible_fruit.n.01", "footwear.n.02"], "f_hard": ["plant.n.02", "animal.n.01", "clothing.n.01"], "f_hard_n": 2, "r_easy": ["boathouse", "mesquite", "rowan"], "r_hard": ["mammal", "conch", "baboon"], "r_hard_n": 2}, {"ent": "jasmine", "cat": "plant.n.02", "cname": "plant", "f_easy": ["clothing.n.01", "aircraft.n.01", "car.n.01"], "f_hard": ["person.n.01", "animal.n.01", "dog.n.01"], "f_hard_n": 2, "r_easy": ["pedophile", "bedroom", "jester"], "r_hard": ["chronicler", "miser", "gasket"], "r_hard_n": 2}, {"ent": "jester", "cat": "person.n.01", "cname": "person", "f_easy": ["furniture.n.01", "machine.n.01", "device.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "room.n.01"], "f_hard_n": 2, "r_easy": ["classroom", "dimmer", "forceps"], "r_hard": ["fuchsia", "bramble", "brazier"], "r_hard_n": 2}, {"ent": "juggler", "cat": "person.n.01", "cname": "person", "f_easy": ["metallic_element.n.01", "animal.n.01", "tree.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "bird.n.01"], "f_hard_n": 2, "r_easy": ["peony", "curry", "bantam"], "r_hard": ["succulent", "starfish", "corgi"], "r_hard_n": 2}, {"ent": "kicker", "cat": "athlete.n.01", "cname": "athlete", "f_easy": ["tree.n.01", "animal.n.01", "material.n.01"], "f_hard": ["artist.n.01", "worker.n.01", "sport.n.01"], "f_hard_n": 2, "r_easy": ["national", "senna", "wanton"], "r_hard": ["commissar", "supermodel", "avenger"], "r_hard_n": 2}, {"ent": "koala", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["dish.n.02", "person.n.01", "clothing.n.01"], "f_hard": ["insect.n.01", "reptile.n.01", "fish.n.01"], "f_hard_n": 3, "r_easy": ["racist", "lodger", "exhibitor"], "r_hard": ["moray", "tortoise", "bumblebee"], "r_hard_n": 3}, {"ent": "krill", "cat": "animal.n.01", "cname": "animal", "f_easy": ["person.n.01", "tree.n.01", "musical_instrument.n.01"], "f_hard": ["person.n.01", "plant.n.02", "plaything.n.01"], "f_hard_n": 2, "r_easy": ["sherry", "hillbilly", "misfit"], "r_hard": ["senna", "miser", "valedictorian"], "r_hard_n": 2}, {"ent": "landlord", "cat": "person.n.01", "cname": "person", "f_easy": ["room.n.01", "footwear.n.02", "equipment.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "material.n.01"], "f_hard_n": 2, "r_easy": ["brogan", "evergreen", "lilac"], "r_hard": ["mammal", "camellia", "classroom"], "r_hard_n": 2}, {"ent": "larva", "cat": "animal.n.01", "cname": "animal", "f_easy": ["machine.n.01", "device.n.01", "container.n.01"], "f_hard": ["plant.n.02", "person.n.01", "vegetable.n.01"], "f_hard_n": 2, "r_easy": ["teapot", "althea", "exporter"], "r_hard": ["babu", "misfit", "locksmith"], "r_hard_n": 2}, {"ent": "liberator", "cat": "person.n.01", "cname": "person", "f_easy": ["equipment.n.01", "mammal.n.01", "tool.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "furniture.n.01"], "f_hard_n": 2, "r_easy": ["mariposa", "palmyra", "minivan"], "r_hard": ["jasmine", "senna", "flamingo"], "r_hard_n": 2}, {"ent": "librarian", "cat": "person.n.01", "cname": "person", "f_easy": ["fish.n.01", "sport.n.01", "edible_fruit.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "reptile.n.01"], "f_hard_n": 2, "r_easy": ["bumblebee", "dreadnought", "bicycle"], "r_hard": ["krill", "maggot", "towel"], "r_hard_n": 2}, {"ent": "lilac", "cat": "plant.n.02", "cname": "plant", "f_easy": ["material.n.01", "game.n.01", "boat.n.01"], "f_hard": ["animal.n.01", "person.n.01", "container.n.01"], "f_hard_n": 2, "r_easy": ["accountant", "mastiff", "jester"], "r_hard": ["wrestler", "dauphin", "messenger"], "r_hard_n": 2}, {"ent": "limousine", "cat": "car.n.01", "cname": "car", "f_easy": ["fish.n.01", "dish.n.02", "fabric.n.01"], "f_hard": ["organ.n.01", "vegetable.n.01", "musical_instrument.n.01"], "f_hard_n": 0, "r_easy": ["spastic", "husband", "vaporizer"], "r_hard": ["betel", "sectional", "rowan"], "r_hard_n": 0}, {"ent": "liza", "cat": "fish.n.01", "cname": "fish", "f_easy": ["aircraft.n.01", "material.n.01", "furniture.n.01"], "f_hard": ["insect.n.01", "bird.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["motorboat", "clockwork", "sabra"], "r_hard": ["woodpecker", "hyena", "mustang"], "r_hard_n": 3}, {"ent": "llama", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["reptile.n.01", "artist.n.01", "artifact.n.01"], "f_hard": ["bird.n.01", "reptile.n.01", "fish.n.01"], "f_hard_n": 3, "r_easy": ["educator", "stepmother", "juggler"], "r_hard": ["hummingbird", "mamba", "crocodile"], "r_hard_n": 3}, {"ent": "locksmith", "cat": "worker.n.01", "cname": "worker", "f_easy": ["device.n.01", "clothing.n.01", "equipment.n.01"], "f_hard": ["athlete.n.01", "artist.n.01", "machine.n.01"], "f_hard_n": 2, "r_easy": ["accused", "raglan", "wraparound"], "r_hard": ["sprinter", "kicker", "granddaughter"], "r_hard_n": 2}, {"ent": "lodger", "cat": "person.n.01", "cname": "person", "f_easy": ["aircraft.n.01", "disease.n.01", "tool.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "musical_instrument.n.01"], "f_hard_n": 2, "r_easy": ["baboon", "bramble", "llama"], "r_hard": ["mariposa", "stray", "azalea"], "r_hard_n": 2}, {"ent": "loom", "cat": "machine.n.01", "cname": "machine", "f_easy": ["vegetable.n.01", "container.n.01", "tool.n.01"], "f_hard": ["musical_instrument.n.01", "weapon.n.01", "clothing.n.01"], "f_hard_n": 2, "r_easy": ["redneck", "cello", "poncho"], "r_hard": ["shiv", "carbine", "paintbrush"], "r_hard_n": 2}, {"ent": "macho", "cat": "person.n.01", "cname": "person", "f_easy": ["fish.n.01", "tree.n.01", "container.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "musical_instrument.n.01"], "f_hard_n": 2, "r_easy": ["scalpel", "goggles", "tavern"], "r_hard": ["krill", "wildflower", "mastiff"], "r_hard_n": 2}, {"ent": "maggot", "cat": "animal.n.01", "cname": "animal", "f_easy": ["edible_fruit.n.01", "metallic_element.n.01", "building.n.01"], "f_hard": ["person.n.01", "plant.n.02", "room.n.01"], "f_hard_n": 2, "r_easy": ["notary", "stagecoach", "turbine"], "r_hard": ["councilman", "statesman", "capacitor"], "r_hard_n": 2}, {"ent": "mahatma", "cat": "person.n.01", "cname": "person", "f_easy": ["artifact.n.01", "car.n.01", "beverage.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "vehicle.n.01"], "f_hard_n": 2, "r_easy": ["ax", "walrus", "dubbing"], "r_hard": ["bot", "starfish", "margarita"], "r_hard_n": 2}, {"ent": "mako", "cat": "fish.n.01", "cname": "fish", "f_easy": ["boat.n.01", "edible_fruit.n.01", "game.n.01"], "f_hard": ["mammal.n.01", "insect.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["underdog", "stapler", "grader"], "r_hard": ["mamba", "woodpecker", "murine"], "r_hard_n": 3}, {"ent": "mamba", "cat": "reptile.n.01", "cname": "reptile", "f_easy": ["material.n.01", "musical_instrument.n.01", "bird.n.01"], "f_hard": ["insect.n.01", "bird.n.01", "fish.n.01"], "f_hard_n": 3, "r_easy": ["recliner", "national", "quitter"], "r_hard": ["mustang", "zebra", "chub"], "r_hard_n": 3}, {"ent": "mammal", "cat": "animal.n.01", "cname": "animal", "f_easy": ["worker.n.01", "tree.n.01", "dish.n.02"], "f_hard": ["person.n.01", "plant.n.02", "tool.n.01"], "f_hard_n": 2, "r_easy": ["cello", "paintbrush", "plenipotentiary"], "r_hard": ["wildflower", "breeder", "dredge"], "r_hard_n": 2}, {"ent": "manger", "cat": "container.n.01", "cname": "container", "f_easy": ["vehicle.n.01", "game.n.01", "building.n.01"], "f_hard": ["building.n.01", "footwear.n.02", "furniture.n.01"], "f_hard_n": 3, "r_easy": ["maggot", "convalescent", "pelican"], "r_hard": ["belting", "sickle", "capacitor"], "r_hard_n": 3}, {"ent": "mangrove", "cat": "tree.n.01", "cname": "tree", "f_easy": ["machine.n.01", "car.n.01", "fish.n.01"], "f_hard": ["flower.n.01", "equipment.n.01", "fish.n.01"], "f_hard_n": 1, "r_easy": ["clary", "penis", "liberator"], "r_hard": ["peony", "mammal", "puppeteer"], "r_hard_n": 1}, {"ent": "marc", "cat": "beverage.n.01", "cname": "drink", "f_easy": ["fabric.n.01", "sport.n.01", "worker.n.01"], "f_hard": ["dish.n.02", "furniture.n.01", "plaything.n.01"], "f_hard_n": 1, "r_easy": ["cardiologist", "condor", "masjid"], "r_hard": ["curry", "sieve", "minion"], "r_hard_n": 1}, {"ent": "margarita", "cat": "beverage.n.01", "cname": "drink", "f_easy": ["building.n.01", "furniture.n.01", "equipment.n.01"], "f_hard": ["dish.n.02", "device.n.01", "person.n.01"], "f_hard_n": 1, "r_easy": ["sectional", "soccer", "gasket"], "r_hard": ["sandwich", "calender", "semaphore"], "r_hard_n": 1}, {"ent": "mariposa", "cat": "plant.n.02", "cname": "plant", "f_easy": ["musical_instrument.n.01", "boat.n.01", "fish.n.01"], "f_hard": ["animal.n.01", "person.n.01", "metallic_element.n.01"], "f_hard_n": 2, "r_easy": ["equatorial", "dermatologist", "socialite"], "r_hard": ["princess", "dauphin", "maggot"], "r_hard_n": 2}, {"ent": "married", "cat": "person.n.01", "cname": "person", "f_easy": ["container.n.01", "game.n.01", "beverage.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "mammal.n.01"], "f_hard_n": 2, "r_easy": ["piglet", "vaporizer", "mesquite"], "r_hard": ["azalea", "mesquite", "conch"], "r_hard_n": 2}, {"ent": "masjid", "cat": "building.n.01", "cname": "building", "f_easy": ["plant.n.02", "athlete.n.01", "boat.n.01"], "f_hard": ["furniture.n.01", "device.n.01", "fabric.n.01"], "f_hard_n": 3, "r_easy": ["contraceptive", "brazier", "handyman"], "r_hard": ["pulley", "handcuff", "ax"], "r_hard_n": 3}, {"ent": "mastiff", "cat": "dog.n.01", "cname": "dog", "f_easy": ["worker.n.01", "equipment.n.01", "material.n.01"], "f_hard": ["device.n.01", "tool.n.01", "body_part.n.01"], "f_hard_n": 0, "r_easy": ["syringe", "heathen", "stapler"], "r_hard": ["brogan", "rifle", "gland"], "r_hard_n": 0}, {"ent": "matador", "cat": "person.n.01", "cname": "person", "f_easy": ["weapon.n.01", "flower.n.01", "food.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "vegetable.n.01"], "f_hard_n": 2, "r_easy": ["calender", "althea", "bobcat"], "r_hard": ["bot", "jasmine", "candlestick"], "r_hard_n": 2}, {"ent": "maul", "cat": "tool.n.01", "cname": "tool", "f_easy": ["person.n.01", "edible_fruit.n.01", "worker.n.01"], "f_hard": ["footwear.n.02", "building.n.01", "furniture.n.01"], "f_hard_n": 3, "r_easy": ["shiv", "cello", "stepmother"], "r_hard": ["masjid", "capacitor", "cesspool"], "r_hard_n": 3}, {"ent": "meatball", "cat": "dish.n.02", "cname": "dish", "f_easy": ["device.n.01", "clothing.n.01", "tool.n.01"], "f_hard": ["beverage.n.01", "dog.n.01", "game.n.01"], "f_hard_n": 1, "r_easy": ["underdog", "liberator", "goggles"], "r_hard": ["sherry", "importer", "desk"], "r_hard_n": 1}, {"ent": "mensch", "cat": "person.n.01", "cname": "person", "f_easy": ["bird.n.01", "tool.n.01", "food.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "sport.n.01"], "f_hard_n": 2, "r_easy": ["dirk", "sandwich", "accelerometer"], "r_hard": ["biennial", "yucca", "fuchsia"], "r_hard_n": 2}, {"ent": "mesquite", "cat": "plant.n.02", "cname": "plant", "f_easy": ["dish.n.02", "dog.n.01", "game.n.01"], "f_hard": ["animal.n.01", "person.n.01", "vehicle.n.01"], "f_hard_n": 2, "r_easy": ["manger", "starfish", "speakeasy"], "r_hard": ["cardiologist", "playmaker", "glider"], "r_hard_n": 2}, {"ent": "messenger", "cat": "person.n.01", "cname": "person", "f_easy": ["body_part.n.01", "edible_fruit.n.01", "machine.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "car.n.01"], "f_hard_n": 2, "r_easy": ["boathouse", "rowan", "washroom"], "r_hard": ["althea", "newt", "thermos"], "r_hard_n": 2}, {"ent": "midshipman", "cat": "person.n.01", "cname": "person", "f_easy": ["metallic_element.n.01", "dog.n.01", "reptile.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "weapon.n.01"], "f_hard_n": 2, "r_easy": ["dinky", "dachshund", "baboon"], "r_hard": ["mammal", "betel", "pelican"], "r_hard_n": 2}, {"ent": "migrant", "cat": "person.n.01", "cname": "person", "f_easy": ["beverage.n.01", "vegetable.n.01", "plant.n.02"], "f_hard": ["animal.n.01", "plant.n.02", "musical_instrument.n.01"], "f_hard_n": 2, "r_easy": ["toby", "windowsill", "recliner"], "r_hard": ["biennial", "yucca", "ratchet"], "r_hard_n": 2}, {"ent": "mimic", "cat": "person.n.01", "cname": "person", "f_easy": ["tool.n.01", "boat.n.01", "machine.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "dish.n.02"], "f_hard_n": 2, "r_easy": ["coffin", "rapier", "sectional"], "r_hard": ["starfish", "azalea", "minivan"], "r_hard_n": 2}, {"ent": "minion", "cat": "person.n.01", "cname": "person", "f_easy": ["metallic_element.n.01", "building.n.01", "aircraft.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "fabric.n.01"], "f_hard_n": 2, "r_easy": ["speakeasy", "carbine", "shirt"], "r_hard": ["newt", "succulent", "shirt"], "r_hard_n": 2}, {"ent": "minivan", "cat": "car.n.01", "cname": "car", "f_easy": ["organ.n.01", "person.n.01", "material.n.01"], "f_hard": ["plaything.n.01", "plant.n.02", "equipment.n.01"], "f_hard_n": 0, "r_easy": ["socialite", "baboon", "pedophile"], "r_hard": ["sled", "environmentalist", "bumblebee"], "r_hard_n": 0}, {"ent": "minnow", "cat": "fish.n.01", "cname": "fish", "f_easy": ["vehicle.n.01", "machine.n.01", "body_part.n.01"], "f_hard": ["bird.n.01", "insect.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["crossbow", "accountant", "crocodile"], "r_hard": ["mamba", "mustang", "walrus"], "r_hard_n": 3}, {"ent": "minster", "cat": "building.n.01", "cname": "building", "f_easy": ["material.n.01", "person.n.01", "furniture.n.01"], "f_hard": ["equipment.n.01", "container.n.01", "vehicle.n.01"], "f_hard_n": 3, "r_easy": ["churn", "environmentalist", "walrus"], "r_hard": ["classroom", "cesspool", "steeper"], "r_hard_n": 3}, {"ent": "miser", "cat": "person.n.01", "cname": "person", "f_easy": ["furniture.n.01", "weapon.n.01", "flower.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "vegetable.n.01"], "f_hard_n": 2, "r_easy": ["thermos", "towel", "sunflower"], "r_hard": ["bramble", "evergreen", "telegraph"], "r_hard_n": 2}, {"ent": "misfit", "cat": "person.n.01", "cname": "person", "f_easy": ["animal.n.01", "boat.n.01", "dish.n.02"], "f_hard": ["animal.n.01", "plant.n.02", "game.n.01"], "f_hard_n": 2, "r_easy": ["washroom", "dreadnought", "bayonet"], "r_hard": ["mesquite", "larva", "speakeasy"], "r_hard_n": 2}, {"ent": "missus", "cat": "person.n.01", "cname": "person", "f_easy": ["musical_instrument.n.01", "room.n.01", "beverage.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "flower.n.01"], "f_hard_n": 2, "r_easy": ["helicopter", "murine", "manger"], "r_hard": ["buttercup", "wildflower", "plow"], "r_hard_n": 2}, {"ent": "modernist", "cat": "artist.n.01", "cname": "artist", "f_easy": ["artifact.n.01", "dish.n.02", "insect.n.01"], "f_hard": ["athlete.n.01", "worker.n.01", "material.n.01"], "f_hard_n": 2, "r_easy": ["papillon", "abolitionist", "chub"], "r_hard": ["boatman", "welder", "telegraph"], "r_hard_n": 2}, {"ent": "moll", "cat": "person.n.01", "cname": "person", "f_easy": ["aircraft.n.01", "tree.n.01", "musical_instrument.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "vehicle.n.01"], "f_hard_n": 2, "r_easy": ["papillon", "rifle", "senna"], "r_hard": ["wildflower", "evergreen", "belting"], "r_hard_n": 2}, {"ent": "mongoose", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["person.n.01", "weapon.n.01", "car.n.01"], "f_hard": ["reptile.n.01", "insect.n.01", "bird.n.01"], "f_hard_n": 3, "r_easy": ["choreographer", "eclectic", "starling"], "r_hard": ["peregrine", "ridley", "stork"], "r_hard_n": 3}, {"ent": "moped", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["flower.n.01", "body_part.n.01", "bird.n.01"], "f_hard": ["building.n.01", "furniture.n.01", "device.n.01"], "f_hard_n": 3, "r_easy": ["watchman", "wraparound", "girder"], "r_hard": ["ax", "transponder", "telescope"], "r_hard_n": 3}, {"ent": "moped", "cat": "container.n.01", "cname": "container", "f_easy": ["building.n.01", "reptile.n.01", "device.n.01"], "f_hard": ["furniture.n.01", "room.n.01", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["valet", "egalitarian", "azalea"], "r_hard": ["sled", "shed", "pulley"], "r_hard_n": 3}, {"ent": "moray", "cat": "fish.n.01", "cname": "fish", "f_easy": ["disease.n.01", "bird.n.01", "organ.n.01"], "f_hard": ["insect.n.01", "reptile.n.01", "bird.n.01"], "f_hard_n": 3, "r_easy": ["harrow", "capo", "aircraft"], "r_hard": ["hummingbird", "rodent", "chimpanzee"], "r_hard_n": 3}, {"ent": "motorboat", "cat": "boat.n.01", "cname": "boat", "f_easy": ["tree.n.01", "machine.n.01", "clothing.n.01"], "f_hard": ["aircraft.n.01", "fabric.n.01", "dog.n.01"], "f_hard_n": 1, "r_easy": ["bistro", "exporter", "beagle"], "r_hard": ["helicopter", "dynamo", "margarita"], "r_hard_n": 1}, {"ent": "motorcycle", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["equipment.n.01", "disease.n.01", "game.n.01"], "f_hard": ["clothing.n.01", "plaything.n.01", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["isolationist", "flashlight", "urchin"], "r_hard": ["syringe", "scalpel", "tavern"], "r_hard_n": 3}, {"ent": "motorcycle", "cat": "container.n.01", "cname": "container", "f_easy": ["artist.n.01", "disease.n.01", "musical_instrument.n.01"], "f_hard": ["tool.n.01", "plaything.n.01", "device.n.01"], "f_hard_n": 3, "r_easy": ["scythe", "newt", "incurable"], "r_hard": ["necktie", "dubbing", "sieve"], "r_hard_n": 3}, {"ent": "murine", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["bird.n.01", "furniture.n.01", "fabric.n.01"], "f_hard": ["bird.n.01", "insect.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["briefcase", "coffin", "huck"], "r_hard": ["condor", "bantam", "finch"], "r_hard_n": 3}, {"ent": "mustang", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["game.n.01", "footwear.n.02", "machine.n.01"], "f_hard": ["bird.n.01", "reptile.n.01", "fish.n.01"], "f_hard_n": 3, "r_easy": ["althea", "amigo", "invalid"], "r_hard": ["condor", "bumblebee", "falcon"], "r_hard_n": 3}, {"ent": "narc", "cat": "person.n.01", "cname": "person", "f_easy": ["clothing.n.01", "furniture.n.01", "fabric.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "furniture.n.01"], "f_hard_n": 2, "r_easy": ["caliper", "vermouth", "rifle"], "r_hard": ["conch", "mesquite", "maggot"], "r_hard_n": 2}, {"ent": "narrator", "cat": "person.n.01", "cname": "person", "f_easy": ["boat.n.01", "body_part.n.01", "room.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "device.n.01"], "f_hard_n": 2, "r_easy": ["cesspool", "walrus", "vise"], "r_hard": ["camellia", "biennial", "classroom"], "r_hard_n": 2}, {"ent": "national", "cat": "person.n.01", "cname": "person", "f_easy": ["clothing.n.01", "building.n.01", "insect.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "clothing.n.01"], "f_hard_n": 2, "r_easy": ["pulley", "fuchsia", "bulldog"], "r_hard": ["conch", "evergreen", "girder"], "r_hard_n": 2}, {"ent": "nawab", "cat": "person.n.01", "cname": "person", "f_easy": ["mammal.n.01", "plaything.n.01", "game.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "metallic_element.n.01"], "f_hard_n": 2, "r_easy": ["motorboat", "starling", "corgi"], "r_hard": ["mammal", "piglet", "genitalia"], "r_hard_n": 2}, {"ent": "necktie", "cat": "clothing.n.01", "cname": "clothing", "f_easy": ["dish.n.02", "plaything.n.01", "car.n.01"], "f_hard": ["building.n.01", "furniture.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["minster", "krill", "psychiatrist"], "r_hard": ["capacitor", "brad", "chalet"], "r_hard_n": 3}, {"ent": "neurologist", "cat": "person.n.01", "cname": "person", "f_easy": ["footwear.n.02", "equipment.n.01", "vegetable.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "equipment.n.01"], "f_hard_n": 2, "r_easy": ["scraper", "greenhouse", "peripheral"], "r_hard": ["lilac", "camellia", "houseboat"], "r_hard_n": 2}, {"ent": "neutral", "cat": "person.n.01", "cname": "person", "f_easy": ["footwear.n.02", "tree.n.01", "aircraft.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "body_part.n.01"], "f_hard_n": 2, "r_easy": ["dubbing", "toby", "ridley"], "r_hard": ["buttercup", "evergreen", "speakeasy"], "r_hard_n": 2}, {"ent": "newsreader", "cat": "person.n.01", "cname": "person", "f_easy": ["game.n.01", "vegetable.n.01", "aircraft.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "dish.n.02"], "f_hard_n": 2, "r_easy": ["suiting", "jasmine", "crocodile"], "r_hard": ["starfish", "camellia", "hooch"], "r_hard_n": 2}, {"ent": "newt", "cat": "animal.n.01", "cname": "animal", "f_easy": ["plant.n.02", "organ.n.01", "musical_instrument.n.01"], "f_hard": ["plant.n.02", "person.n.01", "game.n.01"], "f_hard_n": 2, "r_easy": ["althea", "stepmother", "physiologist"], "r_hard": ["mahatma", "pickpocket", "hooch"], "r_hard_n": 2}, {"ent": "notary", "cat": "worker.n.01", "cname": "worker", "f_easy": ["material.n.01", "musical_instrument.n.01", "artist.n.01"], "f_hard": ["artist.n.01", "athlete.n.01", "beverage.n.01"], "f_hard_n": 2, "r_easy": ["watchman", "succulent", "paisley"], "r_hard": ["kicker", "sprinter", "celibate"], "r_hard_n": 2}, {"ent": "oboe", "cat": "musical_instrument.n.01", "cname": "musical instrument", "f_easy": ["sport.n.01", "beverage.n.01", "organism.n.01"], "f_hard": ["weapon.n.01", "machine.n.01", "dog.n.01"], "f_hard_n": 2, "r_easy": ["starship", "swine", "bodybuilder"], "r_hard": ["gat", "shiv", "spokeswoman"], "r_hard_n": 2}, {"ent": "ombudsman", "cat": "person.n.01", "cname": "person", "f_easy": ["mammal.n.01", "fish.n.01", "clothing.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "boat.n.01"], "f_hard_n": 2, "r_easy": ["mesquite", "bedroom", "genitalia"], "r_hard": ["mammal", "larva", "peony"], "r_hard_n": 2}, {"ent": "overlord", "cat": "person.n.01", "cname": "person", "f_easy": ["fabric.n.01", "tree.n.01", "furniture.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "food.n.01"], "f_hard_n": 2, "r_easy": ["moped", "telescope", "plush"], "r_hard": ["larva", "mariposa", "brogan"], "r_hard_n": 2}, {"ent": "paella", "cat": "dish.n.02", "cname": "dish", "f_easy": ["fish.n.01", "equipment.n.01", "material.n.01"], "f_hard": ["beverage.n.01", "plant.n.02", "fish.n.01"], "f_hard_n": 1, "r_easy": ["prowler", "houseboat", "chalet"], "r_hard": ["brandy", "liberator", "swot"], "r_hard_n": 1}, {"ent": "pagoda", "cat": "building.n.01", "cname": "building", "f_easy": ["dish.n.02", "tree.n.01", "sport.n.01"], "f_hard": ["room.n.01", "fabric.n.01", "container.n.01"], "f_hard_n": 3, "r_easy": ["althea", "newsreader", "pleaser"], "r_hard": ["bedroom", "tartan", "georgette"], "r_hard_n": 3}, {"ent": "paintbrush", "cat": "device.n.01", "cname": "device", "f_easy": ["worker.n.01", "bird.n.01", "artist.n.01"], "f_hard": ["vehicle.n.01", "container.n.01", "fabric.n.01"], "f_hard_n": 3, "r_easy": ["wombat", "raglan", "voyeur"], "r_hard": ["sheeting", "clarence", "dreadnought"], "r_hard_n": 3}, {"ent": "paisley", "cat": "fabric.n.01", "cname": "fabric", "f_easy": ["weapon.n.01", "bird.n.01", "footwear.n.02"], "f_hard": ["equipment.n.01", "footwear.n.02", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["atheist", "stingray", "ethnic"], "r_hard": ["radar", "forceps", "desk"], "r_hard_n": 3}, {"ent": "palmyra", "cat": "tree.n.01", "cname": "tree", "f_easy": ["dog.n.01", "metallic_element.n.01", "food.n.01"], "f_hard": ["flower.n.01", "room.n.01", "athlete.n.01"], "f_hard_n": 1, "r_easy": ["injector", "turbine", "soccer"], "r_hard": ["peony", "cougar", "injector"], "r_hard_n": 1}, {"ent": "panzer", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["animal.n.01", "mammal.n.01", "disease.n.01"], "f_hard": ["furniture.n.01", "building.n.01", "fabric.n.01"], "f_hard_n": 3, "r_easy": ["avenger", "wife", "convert"], "r_hard": ["flashlight", "equatorial", "masjid"], "r_hard_n": 3}, {"ent": "panzer", "cat": "container.n.01", "cname": "container", "f_easy": ["worker.n.01", "device.n.01", "fish.n.01"], "f_hard": ["room.n.01", "equipment.n.01", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["salesperson", "surrealist", "dimmer"], "r_hard": ["tweeter", "boathouse", "incubator"], "r_hard_n": 3}, {"ent": "papillon", "cat": "dog.n.01", "cname": "dog", "f_easy": ["container.n.01", "artist.n.01", "car.n.01"], "f_hard": ["container.n.01", "beverage.n.01", "sport.n.01"], "f_hard_n": 0, "r_easy": ["helicopter", "koala", "peripheral"], "r_hard": ["jester", "preteen", "boatman"], "r_hard_n": 0}, {"ent": "paramedic", "cat": "worker.n.01", "cname": "worker", "f_easy": ["furniture.n.01", "bird.n.01", "animal.n.01"], "f_hard": ["artist.n.01", "athlete.n.01", "food.n.01"], "f_hard_n": 2, "r_easy": ["beekeeper", "clockwork", "bramble"], "r_hard": ["cricketer", "kicker", "chateau"], "r_hard_n": 2}, {"ent": "paranoid", "cat": "person.n.01", "cname": "person", "f_easy": ["machine.n.01", "vegetable.n.01", "fish.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "flower.n.01"], "f_hard_n": 2, "r_easy": ["buttercup", "wombat", "krill"], "r_hard": ["succulent", "fuchsia", "valerian"], "r_hard_n": 2}, {"ent": "pasha", "cat": "person.n.01", "cname": "person", "f_easy": ["insect.n.01", "metallic_element.n.01", "animal.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "food.n.01"], "f_hard_n": 2, "r_easy": ["aircraft", "tavern", "buttercup"], "r_hard": ["succulent", "piglet", "integrator"], "r_hard_n": 2}, {"ent": "patriot", "cat": "person.n.01", "cname": "person", "f_easy": ["plaything.n.01", "footwear.n.02", "game.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "tree.n.01"], "f_hard_n": 2, "r_easy": ["mangrove", "rowan", "peripheral"], "r_hard": ["bot", "camellia", "carbine"], "r_hard_n": 2}, {"ent": "pedophile", "cat": "person.n.01", "cname": "person", "f_easy": ["footwear.n.02", "tool.n.01", "game.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "game.n.01"], "f_hard_n": 2, "r_easy": ["tongs", "forceps", "workstation"], "r_hard": ["mariposa", "clover", "minivan"], "r_hard_n": 2}, {"ent": "pelican", "cat": "bird.n.01", "cname": "bird", "f_easy": ["person.n.01", "building.n.01", "musical_instrument.n.01"], "f_hard": ["insect.n.01", "reptile.n.01", "fish.n.01"], "f_hard_n": 3, "r_easy": ["mongoose", "astrophysicist", "incubator"], "r_hard": ["deer", "baboon", "wombat"], "r_hard_n": 3}, {"ent": "penis", "cat": "organ.n.01", "cname": "organ", "f_easy": ["edible_fruit.n.01", "boat.n.01", "clothing.n.01"], "f_hard": ["artifact.n.01", "dog.n.01", "footwear.n.02"], "f_hard_n": 0, "r_easy": ["camellia", "brad", "absentee"], "r_hard": ["tugboat", "valerian", "messenger"], "r_hard_n": 0}, {"ent": "peony", "cat": "flower.n.01", "cname": "flower", "f_easy": ["game.n.01", "machine.n.01", "aircraft.n.01"], "f_hard": ["tree.n.01", "athlete.n.01", "container.n.01"], "f_hard_n": 1, "r_easy": ["telegraph", "tabor", "maggot"], "r_hard": ["rowan", "stepmother", "visitor"], "r_hard_n": 1}, {"ent": "peregrine", "cat": "bird.n.01", "cname": "bird", "f_easy": ["organ.n.01", "dish.n.02", "plaything.n.01"], "f_hard": ["fish.n.01", "mammal.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["starship", "paramedic", "corgi"], "r_hard": ["swine", "chub", "cheetah"], "r_hard_n": 3}, {"ent": "perfectionist", "cat": "person.n.01", "cname": "person", "f_easy": ["footwear.n.02", "fish.n.01", "vehicle.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "flower.n.01"], "f_hard_n": 2, "r_easy": ["poi", "capacitor", "starling"], "r_hard": ["mammal", "mariposa", "clover"], "r_hard_n": 2}, {"ent": "peripheral", "cat": "equipment.n.01", "cname": "equipment", "f_easy": ["sport.n.01", "athlete.n.01", "vehicle.n.01"], "f_hard": ["device.n.01", "tool.n.01", "building.n.01"], "f_hard_n": 3, "r_easy": ["georgette", "wheelchair", "genitalia"], "r_hard": ["sprinkler", "raglan", "shim"], "r_hard_n": 3}, {"ent": "philanthropist", "cat": "person.n.01", "cname": "person", "f_easy": ["boat.n.01", "metallic_element.n.01", "car.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "building.n.01"], "f_hard_n": 2, "r_easy": ["ax", "gasket", "woodpecker"], "r_hard": ["fuchsia", "wildflower", "dredge"], "r_hard_n": 2}, {"ent": "physicist", "cat": "person.n.01", "cname": "person", "f_easy": ["weapon.n.01", "aircraft.n.01", "dish.n.02"], "f_hard": ["plant.n.02", "animal.n.01", "tree.n.01"], "f_hard_n": 2, "r_easy": ["mastiff", "carbine", "tights"], "r_hard": ["succulent", "betel", "aquarium"], "r_hard_n": 2}, {"ent": "physiologist", "cat": "person.n.01", "cname": "person", "f_easy": ["disease.n.01", "footwear.n.02", "clothing.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "mammal.n.01"], "f_hard_n": 2, "r_easy": ["recliner", "meatball", "contraceptive"], "r_hard": ["althea", "maggot", "murine"], "r_hard_n": 2}, {"ent": "piccolo", "cat": "musical_instrument.n.01", "cname": "musical instrument", "f_easy": ["car.n.01", "plaything.n.01", "person.n.01"], "f_hard": ["machine.n.01", "weapon.n.01", "vegetable.n.01"], "f_hard_n": 2, "r_easy": ["socialite", "raglan", "bantam"], "r_hard": ["gat", "stapler", "slob"], "r_hard_n": 2}, {"ent": "pickpocket", "cat": "person.n.01", "cname": "person", "f_easy": ["body_part.n.01", "plant.n.02", "fish.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "vegetable.n.01"], "f_hard_n": 2, "r_easy": ["semaphore", "handcuff", "starfish"], "r_hard": ["maggot", "senna", "gasket"], "r_hard_n": 2}, {"ent": "piglet", "cat": "animal.n.01", "cname": "animal", "f_easy": ["game.n.01", "dish.n.02", "fabric.n.01"], "f_hard": ["person.n.01", "plant.n.02", "tool.n.01"], "f_hard_n": 2, "r_easy": ["watchman", "playmate", "bodybuilder"], "r_hard": ["fascist", "confidante", "socialite"], "r_hard_n": 2}, {"ent": "pinto", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["person.n.01", "athlete.n.01", "dish.n.02"], "f_hard": ["insect.n.01", "reptile.n.01", "fish.n.01"], "f_hard_n": 3, "r_easy": ["extrovert", "redneck", "workstation"], "r_hard": ["bantam", "liza", "woodpecker"], "r_hard_n": 3}, {"ent": "playmaker", "cat": "person.n.01", "cname": "person", "f_easy": ["dog.n.01", "reptile.n.01", "aircraft.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "mammal.n.01"], "f_hard_n": 2, "r_easy": ["sheeting", "dimmer", "clarence"], "r_hard": ["clary", "larva", "manger"], "r_hard_n": 2}, {"ent": "playmate", "cat": "person.n.01", "cname": "person", "f_easy": ["vehicle.n.01", "fish.n.01", "artifact.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "insect.n.01"], "f_hard_n": 2, "r_easy": ["masjid", "skiff", "mangrove"], "r_hard": ["camellia", "foal", "tavern"], "r_hard_n": 2}, {"ent": "pleaser", "cat": "person.n.01", "cname": "person", "f_easy": ["equipment.n.01", "sport.n.01", "container.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "musical_instrument.n.01"], "f_hard_n": 2, "r_easy": ["mustang", "azalea", "stapler"], "r_hard": ["yucca", "krill", "bulldozer"], "r_hard_n": 2}, {"ent": "plenipotentiary", "cat": "worker.n.01", "cname": "worker", "f_easy": ["athlete.n.01", "artist.n.01", "container.n.01"], "f_hard": ["artist.n.01", "athlete.n.01", "vegetable.n.01"], "f_hard_n": 2, "r_easy": ["rayon", "peregrine", "starship"], "r_hard": ["kicker", "cricketer", "bayonet"], "r_hard_n": 2}, {"ent": "plow", "cat": "tool.n.01", "cname": "tool", "f_easy": ["weapon.n.01", "equipment.n.01", "container.n.01"], "f_hard": ["building.n.01", "clothing.n.01", "device.n.01"], "f_hard_n": 3, "r_easy": ["puppeteer", "commissar", "undergraduate"], "r_hard": ["plush", "tweeter", "thermos"], "r_hard_n": 3}, {"ent": "plumber", "cat": "worker.n.01", "cname": "worker", "f_easy": ["dog.n.01", "game.n.01", "musical_instrument.n.01"], "f_hard": ["athlete.n.01", "artist.n.01", "game.n.01"], "f_hard_n": 2, "r_easy": ["palmyra", "paisley", "dachshund"], "r_hard": ["cricketer", "surrealist", "fancier"], "r_hard_n": 2}, {"ent": "plush", "cat": "fabric.n.01", "cname": "fabric", "f_easy": ["boat.n.01", "vegetable.n.01", "weapon.n.01"], "f_hard": ["room.n.01", "tool.n.01", "furniture.n.01"], "f_hard_n": 3, "r_easy": ["warrior", "pedophile", "coupe"], "r_hard": ["classroom", "raglan", "telescope"], "r_hard_n": 3}, {"ent": "poi", "cat": "dish.n.02", "cname": "dish", "f_easy": ["artist.n.01", "insect.n.01", "organ.n.01"], "f_hard": ["beverage.n.01", "vegetable.n.01", "tool.n.01"], "f_hard_n": 1, "r_easy": ["mesquite", "baboon", "choreographer"], "r_hard": ["whiskey", "chisel", "bantam"], "r_hard_n": 1}, {"ent": "poncho", "cat": "clothing.n.01", "cname": "clothing", "f_easy": ["vehicle.n.01", "room.n.01", "equipment.n.01"], "f_hard": ["plaything.n.01", "vehicle.n.01", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["bicycle", "tricycle", "vise"], "r_hard": ["paintbrush", "brogan", "sprinkler"], "r_hard_n": 3}, {"ent": "pontifical", "cat": "clothing.n.01", "cname": "clothing", "f_easy": ["reptile.n.01", "fabric.n.01", "flower.n.01"], "f_hard": ["container.n.01", "tool.n.01", "vehicle.n.01"], "f_hard_n": 3, "r_easy": ["senator", "wristwatch", "gland"], "r_hard": ["stagecoach", "undercarriage", "shim"], "r_hard_n": 3}, {"ent": "portable", "cat": "machine.n.01", "cname": "machine", "f_easy": ["footwear.n.02", "mammal.n.01", "fish.n.01"], "f_hard": ["weapon.n.01", "musical_instrument.n.01", "boat.n.01"], "f_hard_n": 2, "r_easy": ["trident", "dongle", "valet"], "r_hard": ["oboe", "sarin", "tricycle"], "r_hard_n": 2}, {"ent": "potion", "cat": "beverage.n.01", "cname": "drink", "f_easy": ["artifact.n.01", "vegetable.n.01", "car.n.01"], "f_hard": ["dish.n.02", "equipment.n.01", "machine.n.01"], "f_hard_n": 1, "r_easy": ["caddie", "confidante", "eclectic"], "r_hard": ["sauerkraut", "collie", "panzer"], "r_hard_n": 1}, {"ent": "preteen", "cat": "person.n.01", "cname": "person", "f_easy": ["plant.n.02", "tool.n.01", "food.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "tool.n.01"], "f_hard_n": 2, "r_easy": ["tavern", "belting", "coupe"], "r_hard": ["larva", "clary", "aircraft"], "r_hard_n": 2}, {"ent": "princess", "cat": "person.n.01", "cname": "person", "f_easy": ["device.n.01", "animal.n.01", "edible_fruit.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "disease.n.01"], "f_hard_n": 2, "r_easy": ["shiv", "contraceptive", "papillon"], "r_hard": ["senna", "krill", "towel"], "r_hard_n": 2}, {"ent": "probable", "cat": "person.n.01", "cname": "person", "f_easy": ["musical_instrument.n.01", "food.n.01", "disease.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "body_part.n.01"], "f_hard_n": 2, "r_easy": ["necktie", "bramble", "chub"], "r_hard": ["newt", "evergreen", "manger"], "r_hard_n": 2}, {"ent": "progenitor", "cat": "person.n.01", "cname": "person", "f_easy": ["footwear.n.02", "room.n.01", "mammal.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "container.n.01"], "f_hard_n": 2, "r_easy": ["goggles", "falafel", "vise"], "r_hard": ["bot", "bramble", "stingray"], "r_hard_n": 2}, {"ent": "programmer", "cat": "person.n.01", "cname": "person", "f_easy": ["fabric.n.01", "vegetable.n.01", "sport.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "tree.n.01"], "f_hard_n": 2, "r_easy": ["woodpecker", "chimpanzee", "rapier"], "r_hard": ["clover", "larva", "skiff"], "r_hard_n": 2}, {"ent": "protectionist", "cat": "person.n.01", "cname": "person", "f_easy": ["animal.n.01", "bird.n.01", "flower.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "organ.n.01"], "f_hard_n": 2, "r_easy": ["sunflower", "thermos", "cheetah"], "r_hard": ["bot", "jasmine", "walrus"], "r_hard_n": 2}, {"ent": "prowler", "cat": "person.n.01", "cname": "person", "f_easy": ["clothing.n.01", "aircraft.n.01", "tool.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "container.n.01"], "f_hard_n": 2, "r_easy": ["bulldozer", "gland", "tavern"], "r_hard": ["jasmine", "wildflower", "krill"], "r_hard_n": 2}, {"ent": "psychiatrist", "cat": "person.n.01", "cname": "person", "f_easy": ["material.n.01", "mammal.n.01", "beverage.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "tree.n.01"], "f_hard_n": 2, "r_easy": ["cymbal", "clockwork", "chalet"], "r_hard": ["bramble", "althea", "houseboat"], "r_hard_n": 2}, {"ent": "psychotherapist", "cat": "person.n.01", "cname": "person", "f_easy": ["clothing.n.01", "plaything.n.01", "machine.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "tree.n.01"], "f_hard_n": 2, "r_easy": ["mariposa", "banjo", "dubbing"], "r_hard": ["lilac", "foal", "telegraph"], "r_hard_n": 2}, {"ent": "pug", "cat": "dog.n.01", "cname": "dog", "f_easy": ["building.n.01", "room.n.01", "plant.n.02"], "f_hard": ["plant.n.02", "person.n.01", "plaything.n.01"], "f_hard_n": 0, "r_easy": ["breeder", "meatball", "piglet"], "r_hard": ["ax", "oboe", "tongs"], "r_hard_n": 0}, {"ent": "pulley", "cat": "device.n.01", "cname": "device", "f_easy": ["equipment.n.01", "footwear.n.02", "vegetable.n.01"], "f_hard": ["tool.n.01", "building.n.01", "room.n.01"], "f_hard_n": 3, "r_easy": ["classroom", "plush", "corgi"], "r_hard": ["storeroom", "sarcophagus", "minster"], "r_hard_n": 3}, {"ent": "puppeteer", "cat": "person.n.01", "cname": "person", "f_easy": ["fabric.n.01", "body_part.n.01", "mammal.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "equipment.n.01"], "f_hard_n": 2, "r_easy": ["starship", "undies", "bantam"], "r_hard": ["erica", "foal", "motorcycle"], "r_hard_n": 2}, {"ent": "quitter", "cat": "person.n.01", "cname": "person", "f_easy": ["material.n.01", "flower.n.01", "vegetable.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "fabric.n.01"], "f_hard_n": 2, "r_easy": ["wristwatch", "bicycle", "chateau"], "r_hard": ["wildflower", "larva", "maggot"], "r_hard_n": 2}, {"ent": "racist", "cat": "person.n.01", "cname": "person", "f_easy": ["car.n.01", "mammal.n.01", "disease.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "fabric.n.01"], "f_hard_n": 2, "r_easy": ["workstation", "bedroom", "koala"], "r_hard": ["azalea", "camellia", "gasket"], "r_hard_n": 2}, {"ent": "radar", "cat": "device.n.01", "cname": "device", "f_easy": ["disease.n.01", "insect.n.01", "car.n.01"], "f_hard": ["plaything.n.01", "room.n.01", "clothing.n.01"], "f_hard_n": 3, "r_easy": ["granddaughter", "headmaster", "neutral"], "r_hard": ["huck", "undies", "chateau"], "r_hard_n": 3}, {"ent": "radiologist", "cat": "person.n.01", "cname": "person", "f_easy": ["machine.n.01", "weapon.n.01", "aircraft.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "machine.n.01"], "f_hard_n": 2, "r_easy": ["mariposa", "chateau", "gat"], "r_hard": ["newt", "erica", "goldfish"], "r_hard_n": 2}, {"ent": "raglan", "cat": "clothing.n.01", "cname": "clothing", "f_easy": ["person.n.01", "aircraft.n.01", "tool.n.01"], "f_hard": ["tool.n.01", "vehicle.n.01", "container.n.01"], "f_hard_n": 3, "r_easy": ["clary", "motorcycle", "piccolo"], "r_hard": ["contraceptive", "masjid", "incubator"], "r_hard_n": 3}, {"ent": "rapier", "cat": "weapon.n.01", "cname": "weapon", "f_easy": ["room.n.01", "bird.n.01", "car.n.01"], "f_hard": ["musical_instrument.n.01", "machine.n.01", "furniture.n.01"], "f_hard_n": 2, "r_easy": ["brogan", "avenger", "bot"], "r_hard": ["cymbal", "tambourine", "bicycle"], "r_hard_n": 2}, {"ent": "ratchet", "cat": "device.n.01", "cname": "device", "f_easy": ["edible_fruit.n.01", "dog.n.01", "animal.n.01"], "f_hard": ["container.n.01", "room.n.01", "vehicle.n.01"], "f_hard_n": 3, "r_easy": ["boyfriend", "motorcycle", "physiologist"], "r_hard": ["telegraph", "thermos", "bicycle"], "r_hard_n": 3}, {"ent": "rayon", "cat": "fabric.n.01", "cname": "fabric", "f_easy": ["machine.n.01", "footwear.n.02", "beverage.n.01"], "f_hard": ["device.n.01", "clothing.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["racist", "deflector", "shed"], "r_hard": ["coffin", "chisel", "shim"], "r_hard_n": 3}, {"ent": "recliner", "cat": "furniture.n.01", "cname": "furniture", "f_easy": ["athlete.n.01", "fish.n.01", "beverage.n.01"], "f_hard": ["equipment.n.01", "vehicle.n.01", "clothing.n.01"], "f_hard_n": 3, "r_easy": ["spitfire", "deflector", "hillbilly"], "r_hard": ["peripheral", "sprinkler", "tongs"], "r_hard_n": 3}, {"ent": "redneck", "cat": "person.n.01", "cname": "person", "f_easy": ["vegetable.n.01", "fish.n.01", "bird.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "beverage.n.01"], "f_hard_n": 2, "r_easy": ["stagecoach", "sofa", "mongoose"], "r_hard": ["erica", "maggot", "dirk"], "r_hard_n": 2}, {"ent": "retailer", "cat": "person.n.01", "cname": "person", "f_easy": ["vegetable.n.01", "room.n.01", "sport.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "sport.n.01"], "f_hard_n": 2, "r_easy": ["maggot", "washroom", "tongs"], "r_hard": ["biennial", "betel", "ax"], "r_hard_n": 2}, {"ent": "revisionist", "cat": "person.n.01", "cname": "person", "f_easy": ["metallic_element.n.01", "animal.n.01", "fabric.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "vehicle.n.01"], "f_hard_n": 2, "r_easy": ["azalea", "pug", "briefcase"], "r_hard": ["wildflower", "mesquite", "belting"], "r_hard_n": 2}, {"ent": "rhinoceros", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["weapon.n.01", "person.n.01", "car.n.01"], "f_hard": ["reptile.n.01", "insect.n.01", "bird.n.01"], "f_hard_n": 3, "r_easy": ["motorcycle", "shogun", "cricketer"], "r_hard": ["crocodile", "stork", "minnow"], "r_hard_n": 3}, {"ent": "ridley", "cat": "reptile.n.01", "cname": "reptile", "f_easy": ["beverage.n.01", "organ.n.01", "food.n.01"], "f_hard": ["fish.n.01", "bird.n.01", "insect.n.01"], "f_hard_n": 3, "r_easy": ["supremacist", "crossbow", "statesman"], "r_hard": ["woodpecker", "llama", "pinto"], "r_hard_n": 3}, {"ent": "rifle", "cat": "weapon.n.01", "cname": "weapon", "f_easy": ["fabric.n.01", "food.n.01", "worker.n.01"], "f_hard": ["machine.n.01", "musical_instrument.n.01", "clothing.n.01"], "f_hard_n": 2, "r_easy": ["tabor", "bantam", "starfish"], "r_hard": ["loom", "fife", "dug"], "r_hard_n": 2}, {"ent": "rival", "cat": "person.n.01", "cname": "person", "f_easy": ["sport.n.01", "body_part.n.01", "material.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "room.n.01"], "f_hard_n": 2, "r_easy": ["coupe", "sectional", "deflector"], "r_hard": ["evergreen", "wildflower", "turbine"], "r_hard_n": 2}, {"ent": "rodent", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["vegetable.n.01", "insect.n.01", "tool.n.01"], "f_hard": ["fish.n.01", "reptile.n.01", "insect.n.01"], "f_hard_n": 3, "r_easy": ["tambourine", "puppeteer", "cricketer"], "r_hard": ["moray", "mamba", "flamingo"], "r_hard_n": 3}, {"ent": "rogue", "cat": "person.n.01", "cname": "person", "f_easy": ["sport.n.01", "tree.n.01", "insect.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "fish.n.01"], "f_hard_n": 2, "r_easy": ["conch", "dirk", "whiskey"], "r_hard": ["succulent", "jasmine", "curry"], "r_hard_n": 2}, {"ent": "rowan", "cat": "tree.n.01", "cname": "tree", "f_easy": ["animal.n.01", "worker.n.01", "container.n.01"], "f_hard": ["flower.n.01", "vegetable.n.01", "fabric.n.01"], "f_hard_n": 1, "r_easy": ["abolitionist", "spastic", "buttercup"], "r_hard": ["peony", "accused", "mesquite"], "r_hard_n": 1}, {"ent": "rustic", "cat": "person.n.01", "cname": "person", "f_easy": ["fabric.n.01", "boat.n.01", "plant.n.02"], "f_hard": ["plant.n.02", "animal.n.01", "tool.n.01"], "f_hard_n": 2, "r_easy": ["althea", "maggot", "tricycle"], "r_hard": ["starfish", "senna", "buttermilk"], "r_hard_n": 2}, {"ent": "sabra", "cat": "person.n.01", "cname": "person", "f_easy": ["plaything.n.01", "vegetable.n.01", "device.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "material.n.01"], "f_hard_n": 2, "r_easy": ["candlestick", "bumblebee", "starfish"], "r_hard": ["stray", "starfish", "pinto"], "r_hard_n": 2}, {"ent": "salesperson", "cat": "worker.n.01", "cname": "worker", "f_easy": ["game.n.01", "vehicle.n.01", "plaything.n.01"], "f_hard": ["artist.n.01", "athlete.n.01", "game.n.01"], "f_hard_n": 2, "r_easy": ["flamingo", "wanton", "pleaser"], "r_hard": ["cricketer", "sprinter", "chisel"], "r_hard_n": 2}, {"ent": "sandwich", "cat": "dish.n.02", "cname": "dish", "f_easy": ["edible_fruit.n.01", "game.n.01", "metallic_element.n.01"], "f_hard": ["beverage.n.01", "artist.n.01", "athlete.n.01"], "f_hard_n": 1, "r_easy": ["manger", "liza", "physicist"], "r_hard": ["vermouth", "plumber", "llama"], "r_hard_n": 1}, {"ent": "sarcophagus", "cat": "container.n.01", "cname": "container", "f_easy": ["beverage.n.01", "tree.n.01", "person.n.01"], "f_hard": ["footwear.n.02", "tool.n.01", "clothing.n.01"], "f_hard_n": 3, "r_easy": ["sandwich", "deflector", "psychiatrist"], "r_hard": ["tweeter", "wheelchair", "windowsill"], "r_hard_n": 3}, {"ent": "sarin", "cat": "weapon.n.01", "cname": "weapon", "f_easy": ["furniture.n.01", "animal.n.01", "room.n.01"], "f_hard": ["machine.n.01", "musical_instrument.n.01", "dish.n.02"], "f_hard_n": 2, "r_easy": ["minivan", "astrophysicist", "lilac"], "r_hard": ["turbine", "fife", "washroom"], "r_hard_n": 2}, {"ent": "sauerkraut", "cat": "dish.n.02", "cname": "dish", "f_easy": ["footwear.n.02", "metallic_element.n.01", "person.n.01"], "f_hard": ["beverage.n.01", "plant.n.02", "artist.n.01"], "f_hard_n": 1, "r_easy": ["radiologist", "gland", "sarin"], "r_hard": ["marc", "wraparound", "paisley"], "r_hard_n": 1}, {"ent": "scalpel", "cat": "tool.n.01", "cname": "tool", "f_easy": ["vegetable.n.01", "building.n.01", "musical_instrument.n.01"], "f_hard": ["footwear.n.02", "equipment.n.01", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["puppeteer", "empress", "toby"], "r_hard": ["tricycle", "dongle", "integrator"], "r_hard_n": 3}, {"ent": "scalpel", "cat": "device.n.01", "cname": "device", "f_easy": ["room.n.01", "artist.n.01", "beverage.n.01"], "f_hard": ["fabric.n.01", "footwear.n.02", "room.n.01"], "f_hard_n": 3, "r_easy": ["poi", "undergraduate", "veronica"], "r_hard": ["coffin", "georgette", "moped"], "r_hard_n": 3}, {"ent": "scapegoat", "cat": "person.n.01", "cname": "person", "f_easy": ["dog.n.01", "weapon.n.01", "clothing.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "container.n.01"], "f_hard_n": 2, "r_easy": ["yucca", "steeper", "falcon"], "r_hard": ["evergreen", "stray", "raglan"], "r_hard_n": 2}, {"ent": "scion", "cat": "person.n.01", "cname": "person", "f_easy": ["room.n.01", "artifact.n.01", "tree.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "metallic_element.n.01"], "f_hard_n": 2, "r_easy": ["peripheral", "scythe", "bulldozer"], "r_hard": ["yucca", "betel", "flashlight"], "r_hard_n": 2}, {"ent": "scraper", "cat": "tool.n.01", "cname": "tool", "f_easy": ["fabric.n.01", "building.n.01", "person.n.01"], "f_hard": ["room.n.01", "device.n.01", "container.n.01"], "f_hard_n": 3, "r_easy": ["evergreen", "portable", "hillbilly"], "r_hard": ["telegraph", "storeroom", "paintbrush"], "r_hard_n": 3}, {"ent": "scythe", "cat": "tool.n.01", "cname": "tool", "f_easy": ["material.n.01", "beverage.n.01", "musical_instrument.n.01"], "f_hard": ["container.n.01", "equipment.n.01", "room.n.01"], "f_hard_n": 3, "r_easy": ["exporter", "goggles", "contraceptive"], "r_hard": ["bedroom", "integrator", "tun"], "r_hard_n": 3}, {"ent": "secessionist", "cat": "person.n.01", "cname": "person", "f_easy": ["insect.n.01", "plant.n.02", "musical_instrument.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "plaything.n.01"], "f_hard_n": 2, "r_easy": ["undies", "forceps", "minnow"], "r_hard": ["mammal", "bramble", "fuchsia"], "r_hard_n": 2}, {"ent": "sectional", "cat": "furniture.n.01", "cname": "furniture", "f_easy": ["edible_fruit.n.01", "bird.n.01", "disease.n.01"], "f_hard": ["device.n.01", "footwear.n.02", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["shed", "homophobe", "maggot"], "r_hard": ["equatorial", "sled", "necktie"], "r_hard_n": 3}, {"ent": "semaphore", "cat": "equipment.n.01", "cname": "equipment", "f_easy": ["device.n.01", "worker.n.01", "dog.n.01"], "f_hard": ["plaything.n.01", "device.n.01", "building.n.01"], "f_hard_n": 3, "r_easy": ["conch", "wristwatch", "minnow"], "r_hard": ["girder", "poncho", "bistro"], "r_hard_n": 3}, {"ent": "senator", "cat": "person.n.01", "cname": "person", "f_easy": ["weapon.n.01", "food.n.01", "tool.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "tree.n.01"], "f_hard_n": 2, "r_easy": ["transponder", "sauerkraut", "forceps"], "r_hard": ["foal", "althea", "potion"], "r_hard_n": 2}, {"ent": "senna", "cat": "plant.n.02", "cname": "plant", "f_easy": ["dish.n.02", "beverage.n.01", "animal.n.01"], "f_hard": ["person.n.01", "animal.n.01", "equipment.n.01"], "f_hard_n": 2, "r_easy": ["national", "washroom", "dauphin"], "r_hard": ["lodger", "physiologist", "bumblebee"], "r_hard_n": 2}, {"ent": "shaheed", "cat": "person.n.01", "cname": "person", "f_easy": ["material.n.01", "disease.n.01", "reptile.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "flower.n.01"], "f_hard_n": 2, "r_easy": ["mangrove", "radar", "koala"], "r_hard": ["krill", "conch", "carbine"], "r_hard_n": 2}, {"ent": "shed", "cat": "building.n.01", "cname": "building", "f_easy": ["car.n.01", "metallic_element.n.01", "aircraft.n.01"], "f_hard": ["fabric.n.01", "room.n.01", "furniture.n.01"], "f_hard_n": 3, "r_easy": ["injector", "tweeter", "absentee"], "r_hard": ["tartan", "sprinkler", "pontifical"], "r_hard_n": 3}, {"ent": "sheeting", "cat": "fabric.n.01", "cname": "fabric", "f_easy": ["plant.n.02", "disease.n.01", "metallic_element.n.01"], "f_hard": ["plaything.n.01", "footwear.n.02", "building.n.01"], "f_hard_n": 3, "r_easy": ["wheelchair", "greenhouse", "ombudsman"], "r_hard": ["sofa", "ship", "tavern"], "r_hard_n": 3}, {"ent": "sherry", "cat": "beverage.n.01", "cname": "drink", "f_easy": ["building.n.01", "furniture.n.01", "worker.n.01"], "f_hard": ["dish.n.02", "worker.n.01", "boat.n.01"], "f_hard_n": 1, "r_easy": ["geisha", "undersecretary", "redneck"], "r_hard": ["poi", "carbine", "pleaser"], "r_hard_n": 1}, {"ent": "shim", "cat": "device.n.01", "cname": "device", "f_easy": ["footwear.n.02", "clothing.n.01", "edible_fruit.n.01"], "f_hard": ["container.n.01", "clothing.n.01", "room.n.01"], "f_hard_n": 3, "r_easy": ["bulldozer", "takin", "pug"], "r_hard": ["sled", "semaphore", "sectional"], "r_hard_n": 3}, {"ent": "ship", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["insect.n.01", "worker.n.01", "beverage.n.01"], "f_hard": ["container.n.01", "plaything.n.01", "building.n.01"], "f_hard_n": 3, "r_easy": ["tambourine", "caddie", "tartan"], "r_hard": ["sofa", "churn", "backpack"], "r_hard_n": 3}, {"ent": "shirt", "cat": "clothing.n.01", "cname": "clothing", "f_easy": ["vegetable.n.01", "building.n.01", "dish.n.02"], "f_hard": ["plaything.n.01", "tool.n.01", "room.n.01"], "f_hard_n": 3, "r_easy": ["immigrant", "accused", "sabra"], "r_hard": ["toby", "suiting", "tartan"], "r_hard_n": 3}, {"ent": "shiv", "cat": "weapon.n.01", "cname": "weapon", "f_easy": ["dish.n.02", "vehicle.n.01", "building.n.01"], "f_hard": ["musical_instrument.n.01", "machine.n.01", "food.n.01"], "f_hard_n": 2, "r_easy": ["modernist", "aquarium", "betel"], "r_hard": ["harrow", "cymbal", "portable"], "r_hard_n": 2}, {"ent": "shogun", "cat": "person.n.01", "cname": "person", "f_easy": ["room.n.01", "equipment.n.01", "fabric.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "equipment.n.01"], "f_hard_n": 2, "r_easy": ["portable", "churn", "zebra"], "r_hard": ["conch", "erica", "bulldog"], "r_hard_n": 2}, {"ent": "sickle", "cat": "tool.n.01", "cname": "tool", "f_easy": ["dish.n.02", "mammal.n.01", "fish.n.01"], "f_hard": ["footwear.n.02", "clothing.n.01", "vehicle.n.01"], "f_hard_n": 3, "r_easy": ["sprinkler", "dynamo", "warlord"], "r_hard": ["aircraft", "moped", "shed"], "r_hard_n": 3}, {"ent": "sieve", "cat": "device.n.01", "cname": "device", "f_easy": ["bird.n.01", "person.n.01", "room.n.01"], "f_hard": ["clothing.n.01", "furniture.n.01", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["pinto", "councilman", "homophobe"], "r_hard": ["georgette", "tun", "speakeasy"], "r_hard_n": 3}, {"ent": "skiff", "cat": "boat.n.01", "cname": "boat", "f_easy": ["person.n.01", "equipment.n.01", "fabric.n.01"], "f_hard": ["aircraft.n.01", "building.n.01", "weapon.n.01"], "f_hard_n": 1, "r_easy": ["pulley", "injector", "businesswoman"], "r_hard": ["glider", "incubator", "informer"], "r_hard_n": 1}, {"ent": "sled", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["athlete.n.01", "plant.n.02", "building.n.01"], "f_hard": ["furniture.n.01", "container.n.01", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["rodent", "urchin", "tartan"], "r_hard": ["towel", "windowsill", "tweeter"], "r_hard_n": 3}, {"ent": "slob", "cat": "person.n.01", "cname": "person", "f_easy": ["animal.n.01", "organ.n.01", "plant.n.02"], "f_hard": ["plant.n.02", "animal.n.01", "device.n.01"], "f_hard_n": 2, "r_easy": ["mammal", "penis", "spandex"], "r_hard": ["starfish", "yucca", "wheelchair"], "r_hard_n": 2}, {"ent": "snoop", "cat": "person.n.01", "cname": "person", "f_easy": ["edible_fruit.n.01", "vehicle.n.01", "building.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "musical_instrument.n.01"], "f_hard_n": 2, "r_easy": ["starship", "deer", "motorcycle"], "r_hard": ["clover", "camellia", "llama"], "r_hard_n": 2}, {"ent": "soccer", "cat": "game.n.01", "cname": "game", "f_easy": ["vegetable.n.01", "dog.n.01", "machine.n.01"], "f_hard": ["material.n.01", "disease.n.01", "metallic_element.n.01"], "f_hard_n": 3, "r_easy": ["telescope", "starfish", "chisel"], "r_hard": ["slob", "bulldog", "librarian"], "r_hard_n": 0}, {"ent": "soccer", "cat": "sport.n.01", "cname": "sport", "f_easy": ["car.n.01", "tree.n.01", "organ.n.01"], "f_hard": ["organism.n.01", "vegetable.n.01", "artifact.n.01"], "f_hard_n": 3, "r_easy": ["sheeting", "semaphore", "liza"], "r_hard": ["ombudsman", "untouchable", "breeder"], "r_hard_n": 0}, {"ent": "socialist", "cat": "person.n.01", "cname": "person", "f_easy": ["footwear.n.02", "material.n.01", "metallic_element.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "building.n.01"], "f_hard_n": 2, "r_easy": ["masjid", "dongle", "sofa"], "r_hard": ["biennial", "newt", "krill"], "r_hard_n": 2}, {"ent": "socialite", "cat": "person.n.01", "cname": "person", "f_easy": ["car.n.01", "fish.n.01", "dish.n.02"], "f_hard": ["animal.n.01", "plant.n.02", "container.n.01"], "f_hard_n": 2, "r_easy": ["stork", "moray", "shirt"], "r_hard": ["senna", "evergreen", "fife"], "r_hard_n": 2}, {"ent": "sofa", "cat": "furniture.n.01", "cname": "furniture", "f_easy": ["flower.n.01", "worker.n.01", "body_part.n.01"], "f_hard": ["plaything.n.01", "device.n.01", "container.n.01"], "f_hard_n": 3, "r_easy": ["breeder", "motorboat", "dubbing"], "r_hard": ["tartan", "speakeasy", "chisel"], "r_hard_n": 3}, {"ent": "spandex", "cat": "fabric.n.01", "cname": "fabric", "f_easy": ["mammal.n.01", "tree.n.01", "weapon.n.01"], "f_hard": ["building.n.01", "equipment.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["godmother", "sprinkler", "macho"], "r_hard": ["sprinkler", "desk", "shim"], "r_hard_n": 3}, {"ent": "spastic", "cat": "person.n.01", "cname": "person", "f_easy": ["dog.n.01", "weapon.n.01", "musical_instrument.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "artifact.n.01"], "f_hard_n": 2, "r_easy": ["telescope", "krill", "potion"], "r_hard": ["mariposa", "krill", "dreadnought"], "r_hard_n": 2}, {"ent": "speakeasy", "cat": "room.n.01", "cname": "room", "f_easy": ["tree.n.01", "insect.n.01", "game.n.01"], "f_hard": ["building.n.01", "footwear.n.02", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["chronicler", "patriot", "overlord"], "r_hard": ["brazier", "dropper", "minster"], "r_hard_n": 3}, {"ent": "spitfire", "cat": "person.n.01", "cname": "person", "f_easy": ["vegetable.n.01", "artifact.n.01", "machine.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "building.n.01"], "f_hard_n": 2, "r_easy": ["chub", "plush", "baboon"], "r_hard": ["mesquite", "evergreen", "sandwich"], "r_hard_n": 2}, {"ent": "spokeswoman", "cat": "person.n.01", "cname": "person", "f_easy": ["reptile.n.01", "tool.n.01", "flower.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "dish.n.02"], "f_hard_n": 2, "r_easy": ["desk", "briefcase", "fuchsia"], "r_hard": ["senna", "fuchsia", "aircraft"], "r_hard_n": 2}, {"ent": "sprinkler", "cat": "device.n.01", "cname": "device", "f_easy": ["insect.n.01", "material.n.01", "tool.n.01"], "f_hard": ["clothing.n.01", "room.n.01", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["crocodile", "pontifical", "semaphore"], "r_hard": ["dinky", "chisel", "sheeting"], "r_hard_n": 3}, {"ent": "sprinter", "cat": "athlete.n.01", "cname": "athlete", "f_easy": ["beverage.n.01", "food.n.01", "car.n.01"], "f_hard": ["worker.n.01", "artist.n.01", "device.n.01"], "f_hard_n": 2, "r_easy": ["gat", "sarcophagus", "denture"], "r_hard": ["dragoon", "commissar", "neurologist"], "r_hard_n": 2}, {"ent": "stagecoach", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["worker.n.01", "equipment.n.01", "person.n.01"], "f_hard": ["plaything.n.01", "equipment.n.01", "room.n.01"], "f_hard_n": 3, "r_easy": ["cheetah", "caddie", "brogan"], "r_hard": ["chisel", "aquarium", "pulley"], "r_hard_n": 3}, {"ent": "stagecoach", "cat": "container.n.01", "cname": "container", "f_easy": ["organism.n.01", "plaything.n.01", "fabric.n.01"], "f_hard": ["tool.n.01", "device.n.01", "clothing.n.01"], "f_hard_n": 3, "r_easy": ["wanton", "modernist", "pedophile"], "r_hard": ["sectional", "scythe", "yacht"], "r_hard_n": 3}, {"ent": "stapler", "cat": "machine.n.01", "cname": "machine", "f_easy": ["mammal.n.01", "dish.n.02", "organism.n.01"], "f_hard": ["musical_instrument.n.01", "weapon.n.01", "tool.n.01"], "f_hard_n": 2, "r_easy": ["falcon", "plush", "georgette"], "r_hard": ["shiv", "trident", "marc"], "r_hard_n": 2}, {"ent": "starfish", "cat": "animal.n.01", "cname": "animal", "f_easy": ["container.n.01", "building.n.01", "device.n.01"], "f_hard": ["person.n.01", "plant.n.02", "dish.n.02"], "f_hard_n": 2, "r_easy": ["ratchet", "choreographer", "granddaughter"], "r_hard": ["rival", "misfit", "dropper"], "r_hard_n": 2}, {"ent": "starling", "cat": "bird.n.01", "cname": "bird", "f_easy": ["metallic_element.n.01", "tree.n.01", "furniture.n.01"], "f_hard": ["insect.n.01", "fish.n.01", "mammal.n.01"], "f_hard_n": 3, "r_easy": ["sprinter", "washroom", "radar"], "r_hard": ["walrus", "stingray", "cheetah"], "r_hard_n": 3}, {"ent": "starship", "cat": "equipment.n.01", "cname": "equipment", "f_easy": ["car.n.01", "building.n.01", "food.n.01"], "f_hard": ["furniture.n.01", "fabric.n.01", "device.n.01"], "f_hard_n": 3, "r_easy": ["biennial", "tights", "injector"], "r_hard": ["scythe", "classroom", "washroom"], "r_hard_n": 3}, {"ent": "starship", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["organ.n.01", "device.n.01", "worker.n.01"], "f_hard": ["tool.n.01", "plaything.n.01", "building.n.01"], "f_hard_n": 3, "r_easy": ["buttercup", "bumblebee", "mahatma"], "r_hard": ["girder", "dimmer", "spandex"], "r_hard_n": 3}, {"ent": "statesman", "cat": "person.n.01", "cname": "person", "f_easy": ["fish.n.01", "weapon.n.01", "musical_instrument.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "tool.n.01"], "f_hard_n": 2, "r_easy": ["tugboat", "corgi", "takin"], "r_hard": ["krill", "camellia", "toby"], "r_hard_n": 2}, {"ent": "steamboat", "cat": "boat.n.01", "cname": "boat", "f_easy": ["artist.n.01", "clothing.n.01", "worker.n.01"], "f_hard": ["aircraft.n.01", "metallic_element.n.01", "tool.n.01"], "f_hard_n": 1, "r_easy": ["soccer", "revisionist", "pasha"], "r_hard": ["glider", "miser", "huck"], "r_hard_n": 1}, {"ent": "steeper", "cat": "container.n.01", "cname": "container", "f_easy": ["beverage.n.01", "dog.n.01", "person.n.01"], "f_hard": ["vehicle.n.01", "furniture.n.01", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["chub", "playmate", "extrovert"], "r_hard": ["dimmer", "brad", "shim"], "r_hard_n": 3}, {"ent": "stepdaughter", "cat": "person.n.01", "cname": "person", "f_easy": ["machine.n.01", "fish.n.01", "material.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "mammal.n.01"], "f_hard_n": 2, "r_easy": ["camellia", "minster", "wildflower"], "r_hard": ["jasmine", "piglet", "larva"], "r_hard_n": 2}, {"ent": "stepfather", "cat": "person.n.01", "cname": "person", "f_easy": ["plaything.n.01", "device.n.01", "sport.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "furniture.n.01"], "f_hard_n": 2, "r_easy": ["erica", "starling", "hooch"], "r_hard": ["senna", "bot", "goldfish"], "r_hard_n": 2}, {"ent": "stepmother", "cat": "person.n.01", "cname": "person", "f_easy": ["dish.n.02", "artifact.n.01", "fabric.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "boat.n.01"], "f_hard_n": 2, "r_easy": ["mesquite", "deer", "gavel"], "r_hard": ["yucca", "mammal", "trident"], "r_hard_n": 2}, {"ent": "stingray", "cat": "fish.n.01", "cname": "fish", "f_easy": ["weapon.n.01", "flower.n.01", "edible_fruit.n.01"], "f_hard": ["bird.n.01", "insect.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["nawab", "immune", "clary"], "r_hard": ["takin", "peregrine", "stork"], "r_hard_n": 3}, {"ent": "stockman", "cat": "person.n.01", "cname": "person", "f_easy": ["plaything.n.01", "vehicle.n.01", "metallic_element.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "insect.n.01"], "f_hard_n": 2, "r_easy": ["brandy", "scalpel", "palmyra"], "r_hard": ["jasmine", "newt", "clarence"], "r_hard_n": 2}, {"ent": "stoner", "cat": "person.n.01", "cname": "person", "f_easy": ["building.n.01", "bird.n.01", "plaything.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "device.n.01"], "f_hard_n": 2, "r_easy": ["cesspool", "dinky", "rhinoceros"], "r_hard": ["azalea", "senna", "speakeasy"], "r_hard_n": 2}, {"ent": "storeroom", "cat": "room.n.01", "cname": "room", "f_easy": ["game.n.01", "furniture.n.01", "food.n.01"], "f_hard": ["clothing.n.01", "furniture.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["sabra", "dropper", "husband"], "r_hard": ["suiting", "thermos", "sprinkler"], "r_hard_n": 3}, {"ent": "stork", "cat": "bird.n.01", "cname": "bird", "f_easy": ["car.n.01", "container.n.01", "aircraft.n.01"], "f_hard": ["mammal.n.01", "fish.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["dubbing", "oboe", "toddler"], "r_hard": ["liza", "chipmunk", "mako"], "r_hard_n": 3}, {"ent": "stray", "cat": "animal.n.01", "cname": "animal", "f_easy": ["artist.n.01", "room.n.01", "sport.n.01"], "f_hard": ["plant.n.02", "person.n.01", "plaything.n.01"], "f_hard_n": 2, "r_easy": ["snoop", "waveguide", "composer"], "r_hard": ["amigo", "midshipman", "landlord"], "r_hard_n": 2}, {"ent": "succulent", "cat": "plant.n.02", "cname": "plant", "f_easy": ["beverage.n.01", "machine.n.01", "vegetable.n.01"], "f_hard": ["person.n.01", "animal.n.01", "car.n.01"], "f_hard_n": 2, "r_easy": ["swot", "revisionist", "telegraph"], "r_hard": ["krill", "physicist", "choreographer"], "r_hard_n": 2}, {"ent": "suiting", "cat": "fabric.n.01", "cname": "fabric", "f_easy": ["room.n.01", "plant.n.02", "dish.n.02"], "f_hard": ["plaything.n.01", "building.n.01", "tool.n.01"], "f_hard_n": 3, "r_easy": ["dug", "amigo", "foal"], "r_hard": ["tavern", "thwart", "sickle"], "r_hard_n": 3}, {"ent": "sunflower", "cat": "flower.n.01", "cname": "flower", "f_easy": ["fabric.n.01", "metallic_element.n.01", "person.n.01"], "f_hard": ["tree.n.01", "insect.n.01", "fabric.n.01"], "f_hard_n": 1, "r_easy": ["betrothed", "deer", "shed"], "r_hard": ["mangrove", "bayonet", "gavel"], "r_hard_n": 1}, {"ent": "supermodel", "cat": "worker.n.01", "cname": "worker", "f_easy": ["bird.n.01", "car.n.01", "edible_fruit.n.01"], "f_hard": ["athlete.n.01", "artist.n.01", "plant.n.02"], "f_hard_n": 2, "r_easy": ["hummingbird", "swot", "psychiatrist"], "r_hard": ["sprinter", "surrealist", "playmaker"], "r_hard_n": 2}, {"ent": "supremacist", "cat": "person.n.01", "cname": "person", "f_easy": ["artifact.n.01", "equipment.n.01", "bird.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "device.n.01"], "f_hard_n": 2, "r_easy": ["tugboat", "corgi", "starfish"], "r_hard": ["lilac", "piglet", "llama"], "r_hard_n": 2}, {"ent": "surrealist", "cat": "artist.n.01", "cname": "artist", "f_easy": ["disease.n.01", "organ.n.01", "game.n.01"], "f_hard": ["athlete.n.01", "worker.n.01", "artifact.n.01"], "f_hard_n": 2, "r_easy": ["cutthroat", "warlord", "amplifier"], "r_hard": ["kicker", "notary", "stray"], "r_hard_n": 2}, {"ent": "survivalist", "cat": "person.n.01", "cname": "person", "f_easy": ["plant.n.02", "material.n.01", "aircraft.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "organ.n.01"], "f_hard_n": 2, "r_easy": ["baboon", "walrus", "tartan"], "r_hard": ["succulent", "azalea", "coffin"], "r_hard_n": 2}, {"ent": "swami", "cat": "person.n.01", "cname": "person", "f_easy": ["building.n.01", "plant.n.02", "disease.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "tool.n.01"], "f_hard_n": 2, "r_easy": ["huck", "condor", "clarence"], "r_hard": ["mammal", "erica", "palmyra"], "r_hard_n": 2}, {"ent": "swine", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["furniture.n.01", "bird.n.01", "tree.n.01"], "f_hard": ["bird.n.01", "fish.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["briefcase", "anesthesiologist", "conch"], "r_hard": ["moray", "peregrine", "pelican"], "r_hard_n": 3}, {"ent": "swot", "cat": "person.n.01", "cname": "person", "f_easy": ["disease.n.01", "boat.n.01", "reptile.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "insect.n.01"], "f_hard_n": 2, "r_easy": ["soccer", "falafel", "sled"], "r_hard": ["betel", "bramble", "aquarium"], "r_hard_n": 2}, {"ent": "syringe", "cat": "device.n.01", "cname": "device", "f_easy": ["worker.n.01", "plant.n.02", "car.n.01"], "f_hard": ["room.n.01", "equipment.n.01", "furniture.n.01"], "f_hard_n": 3, "r_easy": ["papillon", "urchin", "bumblebee"], "r_hard": ["rayon", "tun", "sheeting"], "r_hard_n": 3}, {"ent": "tabor", "cat": "musical_instrument.n.01", "cname": "musical instrument", "f_easy": ["food.n.01", "room.n.01", "reptile.n.01"], "f_hard": ["weapon.n.01", "machine.n.01", "dog.n.01"], "f_hard_n": 2, "r_easy": ["debtor", "dirk", "panzer"], "r_hard": ["dredge", "carbine", "prowler"], "r_hard_n": 2}, {"ent": "tailor", "cat": "person.n.01", "cname": "person", "f_easy": ["aircraft.n.01", "organ.n.01", "mammal.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "flower.n.01"], "f_hard_n": 2, "r_easy": ["wraparound", "briefcase", "crocodile"], "r_hard": ["bot", "biennial", "wraparound"], "r_hard_n": 2}, {"ent": "takin", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["person.n.01", "sport.n.01", "disease.n.01"], "f_hard": ["fish.n.01", "insect.n.01", "bird.n.01"], "f_hard_n": 3, "r_easy": ["grocer", "lilac", "socialist"], "r_hard": ["woodpecker", "crocodile", "bumblebee"], "r_hard_n": 3}, {"ent": "tambourine", "cat": "musical_instrument.n.01", "cname": "musical instrument", "f_easy": ["dog.n.01", "boat.n.01", "food.n.01"], "f_hard": ["machine.n.01", "weapon.n.01", "artist.n.01"], "f_hard_n": 2, "r_easy": ["girder", "landlord", "undergraduate"], "r_hard": ["dirk", "dynamo", "mimic"], "r_hard_n": 2}, {"ent": "tartan", "cat": "fabric.n.01", "cname": "fabric", "f_easy": ["equipment.n.01", "organ.n.01", "car.n.01"], "f_hard": ["clothing.n.01", "container.n.01", "room.n.01"], "f_hard_n": 3, "r_easy": ["hummingbird", "vermouth", "dongle"], "r_hard": ["bicycle", "semaphore", "equatorial"], "r_hard_n": 3}, {"ent": "tavern", "cat": "building.n.01", "cname": "building", "f_easy": ["weapon.n.01", "organism.n.01", "insect.n.01"], "f_hard": ["room.n.01", "tool.n.01", "container.n.01"], "f_hard_n": 3, "r_easy": ["socialite", "totalitarian", "mamba"], "r_hard": ["wraparound", "suiting", "capacitor"], "r_hard_n": 3}, {"ent": "teapot", "cat": "container.n.01", "cname": "container", "f_easy": ["equipment.n.01", "reptile.n.01", "worker.n.01"], "f_hard": ["vehicle.n.01", "tool.n.01", "fabric.n.01"], "f_hard_n": 3, "r_easy": ["exporter", "wrestler", "protectionist"], "r_hard": ["candlestick", "desk", "capacitor"], "r_hard_n": 3}, {"ent": "telegraph", "cat": "equipment.n.01", "cname": "equipment", "f_easy": ["fabric.n.01", "person.n.01", "athlete.n.01"], "f_hard": ["fabric.n.01", "plaything.n.01", "clothing.n.01"], "f_hard_n": 3, "r_easy": ["fife", "woodpecker", "bodybuilder"], "r_hard": ["sickle", "pulley", "moped"], "r_hard_n": 3}, {"ent": "telescope", "cat": "device.n.01", "cname": "device", "f_easy": ["sport.n.01", "car.n.01", "edible_fruit.n.01"], "f_hard": ["fabric.n.01", "building.n.01", "container.n.01"], "f_hard_n": 3, "r_easy": ["programmer", "speakeasy", "raglan"], "r_hard": ["bulldozer", "toby", "minster"], "r_hard_n": 3}, {"ent": "therapist", "cat": "person.n.01", "cname": "person", "f_easy": ["weapon.n.01", "musical_instrument.n.01", "artifact.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "equipment.n.01"], "f_hard_n": 2, "r_easy": ["coffin", "harmonica", "dredge"], "r_hard": ["newt", "betel", "yucca"], "r_hard_n": 2}, {"ent": "thermos", "cat": "container.n.01", "cname": "container", "f_easy": ["insect.n.01", "weapon.n.01", "bird.n.01"], "f_hard": ["plaything.n.01", "tool.n.01", "fabric.n.01"], "f_hard_n": 3, "r_easy": ["telegraph", "countess", "dug"], "r_hard": ["storeroom", "transponder", "forceps"], "r_hard_n": 3}, {"ent": "thwart", "cat": "device.n.01", "cname": "device", "f_easy": ["dog.n.01", "boat.n.01", "dish.n.02"], "f_hard": ["plaything.n.01", "footwear.n.02", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["goldfish", "poncho", "wildflower"], "r_hard": ["sickle", "backpack", "moped"], "r_hard_n": 3}, {"ent": "tights", "cat": "clothing.n.01", "cname": "clothing", "f_easy": ["worker.n.01", "furniture.n.01", "dish.n.02"], "f_hard": ["equipment.n.01", "furniture.n.01", "fabric.n.01"], "f_hard_n": 3, "r_easy": ["conch", "thwart", "overlord"], "r_hard": ["contraceptive", "clarence", "telescope"], "r_hard_n": 3}, {"ent": "toby", "cat": "container.n.01", "cname": "container", "f_easy": ["bird.n.01", "body_part.n.01", "disease.n.01"], "f_hard": ["vehicle.n.01", "fabric.n.01", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["claimant", "greyhound", "ancestor"], "r_hard": ["ax", "ratchet", "scythe"], "r_hard_n": 3}, {"ent": "toddler", "cat": "person.n.01", "cname": "person", "f_easy": ["sport.n.01", "musical_instrument.n.01", "edible_fruit.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "organ.n.01"], "f_hard_n": 2, "r_easy": ["foal", "pinto", "coupe"], "r_hard": ["senna", "larva", "potion"], "r_hard_n": 2}, {"ent": "tongs", "cat": "device.n.01", "cname": "device", "f_easy": ["sport.n.01", "animal.n.01", "organ.n.01"], "f_hard": ["furniture.n.01", "fabric.n.01", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["mustang", "narrator", "midshipman"], "r_hard": ["wraparound", "bulldozer", "coffin"], "r_hard_n": 3}, {"ent": "tortoise", "cat": "reptile.n.01", "cname": "reptile", "f_easy": ["material.n.01", "metallic_element.n.01", "artifact.n.01"], "f_hard": ["bird.n.01", "mammal.n.01", "fish.n.01"], "f_hard_n": 3, "r_easy": ["deer", "pasha", "betrothed"], "r_hard": ["hummingbird", "minnow", "moray"], "r_hard_n": 3}, {"ent": "totalitarian", "cat": "person.n.01", "cname": "person", "f_easy": ["sport.n.01", "boat.n.01", "animal.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "edible_fruit.n.01"], "f_hard_n": 2, "r_easy": ["scraper", "buttermilk", "minivan"], "r_hard": ["jasmine", "camellia", "bramble"], "r_hard_n": 2}, {"ent": "towel", "cat": "fabric.n.01", "cname": "fabric", "f_easy": ["material.n.01", "mammal.n.01", "metallic_element.n.01"], "f_hard": ["tool.n.01", "room.n.01", "vehicle.n.01"], "f_hard_n": 3, "r_easy": ["syringe", "alumnus", "genitalia"], "r_hard": ["capacitor", "shirt", "bicycle"], "r_hard_n": 3}, {"ent": "transponder", "cat": "device.n.01", "cname": "device", "f_easy": ["beverage.n.01", "flower.n.01", "building.n.01"], "f_hard": ["plaything.n.01", "container.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["boatman", "paella", "zebra"], "r_hard": ["sarcophagus", "ship", "classroom"], "r_hard_n": 3}, {"ent": "treasurer", "cat": "person.n.01", "cname": "person", "f_easy": ["vehicle.n.01", "body_part.n.01", "flower.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "equipment.n.01"], "f_hard_n": 2, "r_easy": ["thwart", "pinto", "brad"], "r_hard": ["krill", "mammal", "starship"], "r_hard_n": 2}, {"ent": "tricycle", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["artist.n.01", "body_part.n.01", "machine.n.01"], "f_hard": ["plaything.n.01", "furniture.n.01", "clothing.n.01"], "f_hard_n": 3, "r_easy": ["raglan", "urchin", "capo"], "r_hard": ["masjid", "rayon", "scraper"], "r_hard_n": 3}, {"ent": "tricycle", "cat": "container.n.01", "cname": "container", "f_easy": ["sport.n.01", "musical_instrument.n.01", "game.n.01"], "f_hard": ["room.n.01", "device.n.01", "fabric.n.01"], "f_hard_n": 3, "r_easy": ["exhibitor", "steamboat", "chronicler"], "r_hard": ["ax", "plush", "chisel"], "r_hard_n": 3}, {"ent": "trident", "cat": "weapon.n.01", "cname": "weapon", "f_easy": ["boat.n.01", "body_part.n.01", "tree.n.01"], "f_hard": ["machine.n.01", "musical_instrument.n.01", "equipment.n.01"], "f_hard_n": 2, "r_easy": ["rodent", "psychiatrist", "swot"], "r_hard": ["dynamo", "cymbal", "cowgirl"], "r_hard_n": 2}, {"ent": "tugboat", "cat": "boat.n.01", "cname": "boat", "f_easy": ["food.n.01", "person.n.01", "fabric.n.01"], "f_hard": ["aircraft.n.01", "furniture.n.01", "tree.n.01"], "f_hard_n": 1, "r_easy": ["suiting", "perfectionist", "loom"], "r_hard": ["interceptor", "redneck", "brad"], "r_hard_n": 1}, {"ent": "tun", "cat": "container.n.01", "cname": "container", "f_easy": ["room.n.01", "beverage.n.01", "plant.n.02"], "f_hard": ["vehicle.n.01", "building.n.01", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["beagle", "sherry", "veronica"], "r_hard": ["vise", "sectional", "starship"], "r_hard_n": 3}, {"ent": "turbine", "cat": "machine.n.01", "cname": "machine", "f_easy": ["athlete.n.01", "container.n.01", "worker.n.01"], "f_hard": ["musical_instrument.n.01", "weapon.n.01", "tree.n.01"], "f_hard_n": 2, "r_easy": ["newsreader", "migrant", "plumber"], "r_hard": ["trident", "cymbal", "shim"], "r_hard_n": 2}, {"ent": "tweeter", "cat": "device.n.01", "cname": "device", "f_easy": ["fabric.n.01", "insect.n.01", "container.n.01"], "f_hard": ["plaything.n.01", "clothing.n.01", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["warlord", "coupe", "liza"], "r_hard": ["cesspool", "undies", "tartan"], "r_hard_n": 3}, {"ent": "undercarriage", "cat": "device.n.01", "cname": "device", "f_easy": ["beverage.n.01", "vehicle.n.01", "aircraft.n.01"], "f_hard": ["container.n.01", "footwear.n.02", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["sheeting", "grocer", "urchin"], "r_hard": ["storeroom", "tricycle", "panzer"], "r_hard_n": 3}, {"ent": "underdog", "cat": "person.n.01", "cname": "person", "f_easy": ["beverage.n.01", "mammal.n.01", "building.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "disease.n.01"], "f_hard_n": 2, "r_easy": ["sectional", "speakeasy", "chimpanzee"], "r_hard": ["bramble", "bot", "beanie"], "r_hard_n": 2}, {"ent": "undergraduate", "cat": "person.n.01", "cname": "person", "f_easy": ["machine.n.01", "beverage.n.01", "footwear.n.02"], "f_hard": ["animal.n.01", "plant.n.02", "vehicle.n.01"], "f_hard_n": 2, "r_easy": ["fuchsia", "contraceptive", "burrito"], "r_hard": ["succulent", "erica", "wheelchair"], "r_hard_n": 2}, {"ent": "undersecretary", "cat": "person.n.01", "cname": "person", "f_easy": ["container.n.01", "aircraft.n.01", "weapon.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "reptile.n.01"], "f_hard_n": 2, "r_easy": ["meatball", "candlestick", "flamingo"], "r_hard": ["wildflower", "jasmine", "conger"], "r_hard_n": 2}, {"ent": "understudy", "cat": "person.n.01", "cname": "person", "f_easy": ["material.n.01", "tree.n.01", "food.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "building.n.01"], "f_hard_n": 2, "r_easy": ["bot", "sherry", "shim"], "r_hard": ["mariposa", "wildflower", "aircraft"], "r_hard_n": 2}, {"ent": "undies", "cat": "clothing.n.01", "cname": "clothing", "f_easy": ["disease.n.01", "vegetable.n.01", "metallic_element.n.01"], "f_hard": ["fabric.n.01", "building.n.01", "device.n.01"], "f_hard_n": 3, "r_easy": ["stagecoach", "scion", "haggis"], "r_hard": ["churn", "briefcase", "vaporizer"], "r_hard_n": 3}, {"ent": "untouchable", "cat": "person.n.01", "cname": "person", "f_easy": ["disease.n.01", "aircraft.n.01", "device.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "weapon.n.01"], "f_hard_n": 2, "r_easy": ["newt", "brad", "bicycle"], "r_hard": ["fuchsia", "erica", "starling"], "r_hard_n": 2}, {"ent": "urchin", "cat": "person.n.01", "cname": "person", "f_easy": ["fish.n.01", "weapon.n.01", "tree.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "metallic_element.n.01"], "f_hard_n": 2, "r_easy": ["radar", "tambourine", "mongoose"], "r_hard": ["mammal", "wildflower", "speakeasy"], "r_hard_n": 2}, {"ent": "valedictorian", "cat": "person.n.01", "cname": "person", "f_easy": ["food.n.01", "fish.n.01", "material.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "device.n.01"], "f_hard_n": 2, "r_easy": ["syringe", "cello", "dredge"], "r_hard": ["larva", "camellia", "sandwich"], "r_hard_n": 2}, {"ent": "valerian", "cat": "flower.n.01", "cname": "flower", "f_easy": ["fabric.n.01", "dog.n.01", "edible_fruit.n.01"], "f_hard": ["tree.n.01", "aircraft.n.01", "room.n.01"], "f_hard_n": 1, "r_easy": ["brogan", "harmonica", "radiologist"], "r_hard": ["palmyra", "pedophile", "wrestler"], "r_hard_n": 1}, {"ent": "valet", "cat": "worker.n.01", "cname": "worker", "f_easy": ["vehicle.n.01", "artifact.n.01", "equipment.n.01"], "f_hard": ["artist.n.01", "athlete.n.01", "flower.n.01"], "f_hard_n": 2, "r_easy": ["contraceptive", "minivan", "retailer"], "r_hard": ["surrealist", "modernist", "abolitionist"], "r_hard_n": 2}, {"ent": "vaporizer", "cat": "device.n.01", "cname": "device", "f_easy": ["beverage.n.01", "fish.n.01", "flower.n.01"], "f_hard": ["clothing.n.01", "vehicle.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["bumblebee", "piglet", "poncho"], "r_hard": ["sickle", "boathouse", "ship"], "r_hard_n": 3}, {"ent": "vermouth", "cat": "beverage.n.01", "cname": "drink", "f_easy": ["tree.n.01", "organ.n.01", "metallic_element.n.01"], "f_hard": ["dish.n.02", "edible_fruit.n.01", "sport.n.01"], "f_hard_n": 1, "r_easy": ["skiff", "landlord", "aquarium"], "r_hard": ["poi", "swot", "caribou"], "r_hard_n": 1}, {"ent": "veronica", "cat": "flower.n.01", "cname": "flower", "f_easy": ["artist.n.01", "dish.n.02", "machine.n.01"], "f_hard": ["tree.n.01", "insect.n.01", "clothing.n.01"], "f_hard_n": 1, "r_easy": ["thwart", "wanton", "egalitarian"], "r_hard": ["mangrove", "darkroom", "empress"], "r_hard_n": 1}, {"ent": "vise", "cat": "device.n.01", "cname": "device", "f_easy": ["room.n.01", "game.n.01", "beverage.n.01"], "f_hard": ["plaything.n.01", "footwear.n.02", "building.n.01"], "f_hard_n": 3, "r_easy": ["hyena", "physiologist", "bedroom"], "r_hard": ["poncho", "tun", "stagecoach"], "r_hard_n": 3}, {"ent": "visitor", "cat": "person.n.01", "cname": "person", "f_easy": ["equipment.n.01", "machine.n.01", "reptile.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "footwear.n.02"], "f_hard_n": 2, "r_easy": ["hummingbird", "dropper", "bramble"], "r_hard": ["foal", "wildflower", "thwart"], "r_hard_n": 2}, {"ent": "voyeur", "cat": "person.n.01", "cname": "person", "f_easy": ["game.n.01", "reptile.n.01", "room.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "beverage.n.01"], "f_hard_n": 2, "r_easy": ["helicopter", "zebra", "sieve"], "r_hard": ["lilac", "stray", "dreadnought"], "r_hard_n": 2}, {"ent": "waitress", "cat": "worker.n.01", "cname": "worker", "f_easy": ["flower.n.01", "plant.n.02", "dish.n.02"], "f_hard": ["artist.n.01", "athlete.n.01", "fabric.n.01"], "f_hard_n": 2, "r_easy": ["caribou", "totalitarian", "accelerometer"], "r_hard": ["cricketer", "illustrator", "kicker"], "r_hard_n": 2}, {"ent": "walrus", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["aircraft.n.01", "musical_instrument.n.01", "reptile.n.01"], "f_hard": ["reptile.n.01", "bird.n.01", "fish.n.01"], "f_hard_n": 3, "r_easy": ["wristwatch", "sprinkler", "sieve"], "r_hard": ["goldfish", "peregrine", "woodpecker"], "r_hard_n": 3}, {"ent": "wanton", "cat": "person.n.01", "cname": "person", "f_easy": ["container.n.01", "plant.n.02", "fabric.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "reptile.n.01"], "f_hard_n": 2, "r_easy": ["mamba", "plow", "sofa"], "r_hard": ["clover", "krill", "brad"], "r_hard_n": 2}, {"ent": "warden", "cat": "person.n.01", "cname": "person", "f_easy": ["equipment.n.01", "dish.n.02", "vegetable.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "food.n.01"], "f_hard_n": 2, "r_easy": ["buttermilk", "towel", "cello"], "r_hard": ["krill", "camellia", "marc"], "r_hard_n": 2}, {"ent": "warlord", "cat": "person.n.01", "cname": "person", "f_easy": ["beverage.n.01", "boat.n.01", "footwear.n.02"], "f_hard": ["plant.n.02", "animal.n.01", "aircraft.n.01"], "f_hard_n": 2, "r_easy": ["scalpel", "tights", "dynamo"], "r_hard": ["succulent", "starfish", "finch"], "r_hard_n": 2}, {"ent": "warrior", "cat": "person.n.01", "cname": "person", "f_easy": ["body_part.n.01", "boat.n.01", "aircraft.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "vegetable.n.01"], "f_hard_n": 2, "r_easy": ["plow", "mamba", "portable"], "r_hard": ["jasmine", "newt", "cymbal"], "r_hard_n": 2}, {"ent": "washroom", "cat": "room.n.01", "cname": "room", "f_easy": ["game.n.01", "metallic_element.n.01", "reptile.n.01"], "f_hard": ["furniture.n.01", "vehicle.n.01", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["moll", "bantam", "tun"], "r_hard": ["masjid", "peripheral", "integrator"], "r_hard_n": 3}, {"ent": "watchman", "cat": "person.n.01", "cname": "person", "f_easy": ["plaything.n.01", "equipment.n.01", "artifact.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "artifact.n.01"], "f_hard_n": 2, "r_easy": ["cesspool", "motorcycle", "clary"], "r_hard": ["biennial", "yucca", "tabor"], "r_hard_n": 2}, {"ent": "waveguide", "cat": "device.n.01", "cname": "device", "f_easy": ["food.n.01", "container.n.01", "material.n.01"], "f_hard": ["vehicle.n.01", "room.n.01", "plaything.n.01"], "f_hard_n": 3, "r_easy": ["chisel", "soccer", "krill"], "r_hard": ["tun", "teapot", "moped"], "r_hard_n": 3}, {"ent": "weatherman", "cat": "person.n.01", "cname": "person", "f_easy": ["body_part.n.01", "container.n.01", "organ.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "body_part.n.01"], "f_hard_n": 2, "r_easy": ["buggy", "baboon", "dinky"], "r_hard": ["conch", "mariposa", "tights"], "r_hard_n": 2}, {"ent": "welder", "cat": "worker.n.01", "cname": "worker", "f_easy": ["tree.n.01", "bird.n.01", "car.n.01"], "f_hard": ["artist.n.01", "athlete.n.01", "animal.n.01"], "f_hard_n": 2, "r_easy": ["tortoise", "shaheed", "pedophile"], "r_hard": ["modernist", "sprinter", "valedictorian"], "r_hard_n": 2}, {"ent": "wheelchair", "cat": "furniture.n.01", "cname": "furniture", "f_easy": ["disease.n.01", "edible_fruit.n.01", "musical_instrument.n.01"], "f_hard": ["room.n.01", "device.n.01", "clothing.n.01"], "f_hard_n": 3, "r_easy": ["revisionist", "narrator", "maggot"], "r_hard": ["brazier", "steeper", "plow"], "r_hard_n": 3}, {"ent": "whiskey", "cat": "beverage.n.01", "cname": "drink", "f_easy": ["animal.n.01", "worker.n.01", "flower.n.01"], "f_hard": ["dish.n.02", "tool.n.01", "vegetable.n.01"], "f_hard_n": 1, "r_easy": ["glider", "retailer", "dauphin"], "r_hard": ["haggis", "sofa", "mustang"], "r_hard_n": 1}, {"ent": "wife", "cat": "person.n.01", "cname": "person", "f_easy": ["furniture.n.01", "edible_fruit.n.01", "plant.n.02"], "f_hard": ["animal.n.01", "plant.n.02", "equipment.n.01"], "f_hard_n": 2, "r_easy": ["liza", "thermos", "maggot"], "r_hard": ["senna", "succulent", "dug"], "r_hard_n": 2}, {"ent": "wildflower", "cat": "plant.n.02", "cname": "plant", "f_easy": ["edible_fruit.n.01", "fabric.n.01", "plaything.n.01"], "f_hard": ["person.n.01", "animal.n.01", "mammal.n.01"], "f_hard_n": 2, "r_easy": ["scapegoat", "swami", "zebra"], "r_hard": ["rogue", "warlord", "stingray"], "r_hard_n": 2}, {"ent": "windowsill", "cat": "device.n.01", "cname": "device", "f_easy": ["dish.n.02", "vehicle.n.01", "building.n.01"], "f_hard": ["vehicle.n.01", "container.n.01", "equipment.n.01"], "f_hard_n": 3, "r_easy": ["margarita", "mimic", "psychotherapist"], "r_hard": ["washroom", "buggy", "cesspool"], "r_hard_n": 3}, {"ent": "wombat", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["building.n.01", "machine.n.01", "worker.n.01"], "f_hard": ["reptile.n.01", "bird.n.01", "insect.n.01"], "f_hard_n": 3, "r_easy": ["rapier", "watchman", "brad"], "r_hard": ["goldfish", "stingray", "conger"], "r_hard_n": 3}, {"ent": "woodpecker", "cat": "bird.n.01", "cname": "bird", "f_easy": ["device.n.01", "vegetable.n.01", "tool.n.01"], "f_hard": ["fish.n.01", "mammal.n.01", "insect.n.01"], "f_hard_n": 3, "r_easy": ["messenger", "housemate", "senna"], "r_hard": ["murine", "rodent", "wombat"], "r_hard_n": 3}, {"ent": "workstation", "cat": "machine.n.01", "cname": "machine", "f_easy": ["dog.n.01", "artist.n.01", "tool.n.01"], "f_hard": ["weapon.n.01", "musical_instrument.n.01", "tree.n.01"], "f_hard_n": 2, "r_easy": ["shogun", "huck", "sectional"], "r_hard": ["bayonet", "sarin", "chalet"], "r_hard_n": 2}, {"ent": "worthy", "cat": "person.n.01", "cname": "person", "f_easy": ["machine.n.01", "plaything.n.01", "car.n.01"], "f_hard": ["animal.n.01", "plant.n.02", "tool.n.01"], "f_hard_n": 2, "r_easy": ["maul", "tweeter", "hooch"], "r_hard": ["clary", "succulent", "dug"], "r_hard_n": 2}, {"ent": "wraparound", "cat": "clothing.n.01", "cname": "clothing", "f_easy": ["insect.n.01", "body_part.n.01", "aircraft.n.01"], "f_hard": ["vehicle.n.01", "equipment.n.01", "furniture.n.01"], "f_hard_n": 3, "r_easy": ["attacker", "turbine", "peripheral"], "r_hard": ["brogan", "chateau", "gasket"], "r_hard_n": 3}, {"ent": "wrestler", "cat": "person.n.01", "cname": "person", "f_easy": ["beverage.n.01", "material.n.01", "dog.n.01"], "f_hard": ["plant.n.02", "animal.n.01", "dog.n.01"], "f_hard_n": 2, "r_easy": ["classroom", "evergreen", "aircraft"], "r_hard": ["newt", "buttercup", "dubbing"], "r_hard_n": 2}, {"ent": "wristwatch", "cat": "device.n.01", "cname": "device", "f_easy": ["worker.n.01", "bird.n.01", "organ.n.01"], "f_hard": ["clothing.n.01", "equipment.n.01", "footwear.n.02"], "f_hard_n": 3, "r_easy": ["delinquent", "yacht", "shogun"], "r_hard": ["maul", "buggy", "beanie"], "r_hard_n": 3}, {"ent": "yacht", "cat": "vehicle.n.01", "cname": "vehicle", "f_easy": ["room.n.01", "clothing.n.01", "equipment.n.01"], "f_hard": ["equipment.n.01", "plaything.n.01", "device.n.01"], "f_hard_n": 3, "r_easy": ["huck", "rival", "paramedic"], "r_hard": ["peripheral", "spandex", "cesspool"], "r_hard_n": 3}, {"ent": "yucca", "cat": "plant.n.02", "cname": "plant", "f_easy": ["device.n.01", "fabric.n.01", "musical_instrument.n.01"], "f_hard": ["person.n.01", "animal.n.01", "organ.n.01"], "f_hard_n": 2, "r_easy": ["workstation", "hangman", "migrant"], "r_hard": ["philanthropist", "redneck", "undercarriage"], "r_hard_n": 2}, {"ent": "zebra", "cat": "mammal.n.01", "cname": "mammal", "f_easy": ["plaything.n.01", "musical_instrument.n.01", "disease.n.01"], "f_hard": ["insect.n.01", "bird.n.01", "reptile.n.01"], "f_hard_n": 3, "r_easy": ["sauerkraut", "headmaster", "spitfire"], "r_hard": ["bantam", "hummingbird", "mamba"], "r_hard_n": 3}]}')
ITEMS, CATS, ENTS = DATA['items'], DATA['cats'], DATA['ents']
MC = mc_set(ITEMS, CATS)
print('4択の問い', len(MC)); print(MC[0]['q']); print(); print(MC[-1]['q'])

## 1. Gemma に4択で聞く（5〜10分）

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
gm = AutoModelForCausalLM.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), torch_dtype=torch.float16, device_map="cuda").eval()
chat = lambda q: tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
LT = [tok.encode(c, add_special_tokens=False)[0] for c in LET]
assert len(set(LT)) == 4
def letters(qs, B=32):
    out = []; t = time.time()
    for i in range(0, len(qs), B):
        enc = tok([chat(q) for q in qs[i:i + B]], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            r = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
        out.extend(torch.log_softmax(r[:, LT], -1).cpu().numpy().round(4).tolist()); del r
        if (i // B) % 50 == 0: print(f"  {i}/{len(qs)}  {time.time() - t:.0f}秒")
    return out
LP = letters([m["q"] for m in MC])

## 2. 自由に書かせる（5分）

In [ ]:
def write(qs, n_new, B=32):
    out = []
    for i in range(0, len(qs), B):
        enc = tok([chat(q) for q in qs[i:i + B]], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            g = gm.generate(**enc, max_new_tokens=n_new, do_sample=False)
        out.extend(tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True))
    return out
OF = write([open_f_prompt(x) for x in ENTS], 10)
OR = write([open_r_prompt(CATS[c]) for c in CATS], 240, B=16)
print(ENTS[0], "→", OF[0]); print(list(CATS.values())[0], "→", parse_list(OR[0])[:8])

## 3. 自由に書いた答えを WordNet で確かめる（参考。WordNet の採点は狭いことがある）

In [ ]:
def syns(w):
    return wn.synsets(w.strip().replace(" ", "_"), pos=wn.NOUN)
def ups(s):
    return set(s.closure(lambda z: z.hypernyms() + z.instance_hypernyms())) | {s}
def f_ok(ent, ans):
    a = ans.strip().strip(".").lower()
    cand = syns(a) or syns(a.split()[-1]) if a else []
    U = set().union(*[ups(s) - {s} for s in syns(ent)]) if syns(ent) else set()
    return bool(set(cand) & U)
def r_ok(cat, word):
    C = wn.synset(cat)
    return any(C in ups(s) and s != C for s in (syns(word) or (syns(word.split()[-1]) if word.split() else [])))
F_OK = [f_ok(x, a) for x, a in zip(ENTS, OF)]
R_LIST = [parse_list(t) for t in OR]
R_OK = [[r_ok(c, w) for w in L] for c, L in zip(CATS, R_LIST)]
print("前向き 自由回答の正しさ", round(float(np.mean(F_OK)), 3))
print("逆向き 自由回答の正しさ（挙げた物のうち）", round(float(np.mean([v for L in R_OK for v in L])), 3), " 1分類あたり挙げた数", round(float(np.mean([len(L) for L in R_LIST])), 1))

## 4. 保存と速報

In [ ]:
res = dict(model=GEMMA_ID, mc=[dict(m, lp=l) for m, l in zip(MC, LP)], open_f=OF, open_r=OR, f_ok=F_OK, r_list=R_LIST, r_ok=R_OK)
for m in res["mc"]: m.pop("q")
with gzip.open(f"{OUT}/E160_results.json.gz", "wt", encoding="utf-8") as f: json.dump(res, f, ensure_ascii=False)
print("保存しました", OUT)
acc = {}
for m in res["mc"]:
    acc.setdefault((m["d"], m["lv"]), []).append(int(np.argmax(m["lp"]) == m["pos"]))
for k in sorted(acc): print("前向き" if k[0] == "f" else "逆向き", "やさしい" if k[1] == "easy" else "難しい", round(float(np.mean(acc[k])), 3))